In [ ]:
import re
import json
import os
import shutil
import gc
import traceback
from collections import Counter
from langchain.document_loaders import PyMuPDFLoader
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

class SentenceTransformerEmbedding:
    """
    Wrapper for SentenceTransformer to provide embed_documents and embed_query methods.
    """
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name, trust_remote_code=True)

    def embed_documents(self, texts):
        try:
            return self.model.encode(texts, batch_size=10).tolist()
        except Exception as e:
            print("❌ [임베딩 오류]:", e)
            raise

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

def clean_legal_text(text):
    text = re.sub(r'^\s*법제처.*국가법령정보센터\s*$|^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\(.*?\)\s*\d{2,3}-\d{3,4}-\d{4}', '', text)
    text = re.sub(r'(\n\s*){2,}', '\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'^\s*\[.*?\]$', '', text, flags=re.MULTILINE)
    return text.strip()

def extract_revision_dates(segment):
    revision_pattern = r"(\d{4})\.\s*(\d{1,2})\.\s*(\d{1,2})"
    matches = re.findall(revision_pattern, segment)
    return [{"year": int(year), "month": int(month), "day": int(day)} for year, month, day in matches]

def simplify_related_references(related_refs):
    simplified_refs = []
    for ref in related_refs:
        top_level = re.match(r'(제\d+조)', ref)
        if top_level:
            simplified_refs.append(top_level.group(1))
    ref_counts = Counter(simplified_refs)
    sorted_refs = sorted(ref_counts.items(), key=lambda x: (-x[1], x[0]))
    return [ref for ref, _ in sorted_refs]

def extract_metadata(segment):
    article_match = re.search(r'(제\s*\d+\s*조)\s*\((.*?)\)', segment)
    related_refs = re.findall(r'제\d+조(?:제\d+항)?(?:제\d+호)?', segment)
    simplified_refs = simplify_related_references(related_refs)
    related_refs_str = ", ".join(simplified_refs) if simplified_refs else "Not Applicable"
    revision_dates = extract_revision_dates(segment)
    revision_dates_str = json.dumps(revision_dates) if revision_dates else "Not Applicable"
    return {
        "article": article_match.group(1) if article_match else "Unknown",
        "title": article_match.group(2) if article_match else "Unknown",
        "related_references": related_refs_str,
        "revision_dates": revision_dates_str
    }

def remove_duplicates(documents):
    unique_content = {}
    unique_documents = []
    for doc in documents:
        if doc.page_content not in unique_content:
            unique_content[doc.page_content] = doc.metadata
            unique_documents.append(doc)
        else:
            unique_metadata = unique_content[doc.page_content]
            unique_metadata.update(doc.metadata)
            unique_content[doc.page_content] = unique_metadata
    return [
        Document(page_content=content, metadata=meta)
        for content, meta in unique_content.items()
    ]

def keyword_based_splitter(text, pattern=r'제\s*\d+\s*조\s*\(.*?\)'):
    matches = re.finditer(pattern, text)
    split_positions = [match.start() for match in matches]
    split_positions.append(len(text))
    segments = []
    for i in range(len(split_positions) - 1):
        start = split_positions[i]
        end = split_positions[i + 1]
        segments.append(text[start:end].strip())
    return segments

def reset_chroma_directory(directory):
    if os.path.exists(directory):
        shutil.rmtree(directory)

def process_and_store_pdf_in_chroma(directory_path, pattern=r'제\s*\d+\s*조\s*\(.*?\)', chroma_dir="chroma_data"):
    print("▶ [START] Chroma 디렉토리 초기화")
    reset_chroma_directory(chroma_dir)

    print("▶ [START] 임베딩 모델 초기화")
    embedding_model = SentenceTransformerEmbedding("jinaai/jina-embeddings-v3")
    vectorstore = None

    for file_name in os.listdir(directory_path):
        if not file_name.endswith(".pdf"):
            continue
        file_path = os.path.join(directory_path, file_name)
        print(f"\n📄 [PROCESSING FILE] {file_path}")

        try:
            print("  ⮕ PyMuPDFLoader 시작")
            loader = PyMuPDFLoader(file_path)
            documents = loader.load()
            print(f"  ⮕ 로드된 문서 페이지 수: {len(documents)}")

            all_document_objects = []

            for i, doc in enumerate(documents):
                try:
                    print(f"    ├─ [PAGE {i+1}] 텍스트 정리 시작")
                    cleaned_text = clean_legal_text(doc.page_content)
                    if not cleaned_text.strip():
                        print(f"    ├─ [SKIP] 빈 페이지 내용 (page {i+1})")
                        continue

                    print("    ├─ 텍스트 분할 시작")
                    split_segments = keyword_based_splitter(cleaned_text, pattern)
                    document_objects = []

                    for j, segment in enumerate(split_segments):
                        if not segment.strip():
                            print(f"    ⚠️ [SKIP] 빈 segment 발견 (page {i+1}, segment {j+1})")
                            continue

                        metadata = extract_metadata(segment)
                        metadata["source"] = file_path
                        document_objects.append(Document(page_content=segment, metadata=metadata))

                    document_objects = remove_duplicates(document_objects)
                    all_document_objects.extend(document_objects)

                except Exception as page_err:
                    print(f"    ❌ [ERROR] 페이지 {i+1} 처리 중 오류:\n{traceback.format_exc()}")

                finally:
                    del cleaned_text, split_segments, document_objects
                    gc.collect()

            if not all_document_objects:
                print(f"  ⚠️ [SKIP] 유효한 segment 없음, 건너뜀: {file_path}")
                continue

            print(f"  ⮕ 벡터스토어에 문서 추가 (총 {len(all_document_objects)}개)")

            if vectorstore is None:
                print("  ⮕ 초기 벡터스토어 생성 중 (첫 배치)")
                vectorstore = Chroma.from_documents(
                    documents=[],
                    embedding=embedding_model,
                    persist_directory=chroma_dir
                )

            # 🔹 배치로 문서 추가
            batch_size = 10
            for i in range(0, len(all_document_objects), batch_size):
                batch = all_document_objects[i:i+batch_size]
                print(f"    ⏳ [벡터 추가 중] 문서 {i+1}~{i+len(batch)}")
                vectorstore.add_documents(batch)
                gc.collect()

        except Exception as file_err:
            print(f"❌ [FATAL] 파일 처리 중 오류 발생 ({file_path}):\n{traceback.format_exc()}")

        finally:
            del all_document_objects, documents
            gc.collect()

    print("▶ [SAVE] 벡터스토어 저장 시작")
    if vectorstore:
        vectorstore.persist()
        print("✅ [DONE] 저장 완료.")
    else:
        print("⚠️ [SKIP] 저장할 벡터가 없음.")
    return vectorstore

In [ ]:
directory_path = "file/도로 교통법"
chroma_dir = "chroma_data_folder"

vectorstore = process_and_store_pdf_in_chroma(directory_path, chroma_dir=chroma_dir)
retriever = vectorstore.as_retriever(search_kwargs={'k': 10})
retriever.get_relevant_documents("교통안전")

In [ ]:
from langchain.document_loaders import PyMuPDFLoader

def test_pdf_loader(file_path, num_pages=3):
    # 1) PDF 로드
    loader = PyMuPDFLoader(file_path)
    docs = loader.load()
    print(f"✅ 로드된 페이지 수: {len(docs)}")

    # 2) 지정한 페이지만 샘플 출력
    for i, doc in enumerate(docs[:num_pages]):
        print(f"\n--- Page {i+1} (길이: {len(doc.page_content)}자) ---")
        print(doc.page_content[:500])  # 앞 500자만 출력

if __name__ == "__main__":
    pdf_path = "file/도로 교통법/도로교통법 시행규칙(행정안전부령)(제00507호)(20240731).pdf"
    test_pdf_loader(pdf_path, num_pages=5)

In [ ]:
import re
import json
import os
import shutil
import gc
import traceback
from collections import Counter
from langchain.document_loaders import PyMuPDFLoader
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

class SentenceTransformerEmbedding:
    """
    Wrapper for SentenceTransformer to provide embed_documents and embed_query methods.
    """
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name, trust_remote_code=True)

    def embed_documents(self, texts):
        try:
            return self.model.encode(texts, batch_size=10).tolist()
        except Exception as e:
            print("❌ [임베딩 오류]:", e)
            raise

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

def clean_legal_text(text):
    text = re.sub(r'^\s*법제처.*국가법령정보센터\s*$|^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\(.*?\)\s*\d{2,3}-\d{3,4}-\d{4}', '', text)
    text = re.sub(r'(\n\s*){2,}', '\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'^\s*\[.*?\]$', '', text, flags=re.MULTILINE)
    return text.strip()

def extract_revision_dates(segment):
    revision_pattern = r"(\d{4})\.\s*(\d{1,2})\.\s*(\d{1,2})"
    matches = re.findall(revision_pattern, segment)
    return [{"year": int(year), "month": int(month), "day": int(day)} for year, month, day in matches]

def simplify_related_references(related_refs):
    simplified_refs = []
    for ref in related_refs:
        top_level = re.match(r'(제\d+조)', ref)
        if top_level:
            simplified_refs.append(top_level.group(1))
    ref_counts = Counter(simplified_refs)
    sorted_refs = sorted(ref_counts.items(), key=lambda x: (-x[1], x[0]))
    return [ref for ref, _ in sorted_refs]

def extract_metadata(segment):
    article_match = re.search(r'(제\s*\d+\s*조)\s*\((.*?)\)', segment)
    related_refs = re.findall(r'제\d+조(?:제\d+항)?(?:제\d+호)?', segment)
    simplified_refs = simplify_related_references(related_refs)
    related_refs_str = ", ".join(simplified_refs) if simplified_refs else "Not Applicable"
    revision_dates = extract_revision_dates(segment)
    revision_dates_str = json.dumps(revision_dates) if revision_dates else "Not Applicable"
    return {
        "article": article_match.group(1) if article_match else "Unknown",
        "title": article_match.group(2) if article_match else "Unknown",
        "related_references": related_refs_str,
        "revision_dates": revision_dates_str
    }

def remove_duplicates(documents):
    unique_content = {}
    unique_documents = []
    for doc in documents:
        if doc.page_content not in unique_content:
            unique_content[doc.page_content] = doc.metadata
            unique_documents.append(doc)
        else:
            unique_metadata = unique_content[doc.page_content]
            unique_metadata.update(doc.metadata)
            unique_content[doc.page_content] = unique_metadata
    return [Document(page_content=content, metadata=meta) for content, meta in unique_content.items()]

def keyword_based_splitter(text, pattern=r'제\s*\d+\s*조\s*\(.*?\)'):
    matches = re.finditer(pattern, text)
    split_positions = [match.start() for match in matches]
    split_positions.append(len(text))
    segments = []
    for i in range(len(split_positions) - 1):
        start = split_positions[i]
        end = split_positions[i + 1]
        segments.append(text[start:end].strip())
    return segments

def reset_chroma_directory(directory):
    if os.path.exists(directory):
        shutil.rmtree(directory)

def process_and_store_pdf_in_chroma(directory_path, pattern=r'제\s*\d+\s*조\s*\(.*?\)', chroma_dir="chroma_data"):
    print("▶ [START] Chroma 디렉토리 초기화")
    reset_chroma_directory(chroma_dir)

    print("▶ [START] 임베딩 모델 초기화")
    embedding_model = SentenceTransformerEmbedding("jinaai/jina-embeddings-v3")
    vectorstore = None

    for file_name in os.listdir(directory_path):
        if not file_name.endswith(".pdf"):
            continue
        file_path = os.path.join(directory_path, file_name)
        print(f"\n📄 [PROCESSING FILE] {file_path}")

        try:
            print("  ⮕ PyMuPDFLoader 시작")
            loader = PyMuPDFLoader(file_path)
            documents = loader.load()
            print(f"  ⮕ 로드된 문서 페이지 수: {len(documents)}")

            all_document_objects = []

            for i, doc in enumerate(documents):
                try:
                    print(f"    ├─ [PAGE {i+1}] 텍스트 정리 시작")
                    cleaned_text = clean_legal_text(doc.page_content)
                    if not cleaned_text.strip():
                        print(f"    ├─ [SKIP] 빈 페이지 내용 (page {i+1})")
                        continue

                    print("    ├─ 텍스트 분할 시작")
                    split_segments = keyword_based_splitter(cleaned_text, pattern)
                    document_objects = []

                    for j, segment in enumerate(split_segments):
                        # 빈 segment 제거
                        if not segment.strip():
                            print(f"    ⚠️ [SKIP] 빈 segment 발견 (page {i+1}, segment {j+1})")
                            continue

                        metadata = extract_metadata(segment)
                        metadata["source"] = file_path
                        document_objects.append(Document(page_content=segment, metadata=metadata))

                    document_objects = remove_duplicates(document_objects)
                    all_document_objects.extend(document_objects)

                except Exception as page_err:
                    print(f"    ❌ [ERROR] 페이지 {i+1} 처리 중 오류:\n{traceback.format_exc()}")

                finally:
                    del cleaned_text, split_segments, document_objects
                    gc.collect()

            if not all_document_objects:
                print(f"  ⚠️ [SKIP] 유효한 segment 없음, 건너뜀: {file_path}")
                continue

            print(f"  ⮕ 벡터스토어에 문서 추가 (총 {len(all_document_objects)}개)")

            if vectorstore is None:
                print("  ⮕ 초기 벡터스토어 생성 중 (첫 배치)")
                vectorstore = Chroma.from_documents(
                    documents=[],
                    embedding=embedding_model,
                    persist_directory=chroma_dir
                )

            batch_size = 10
            for idx in range(0, len(all_document_objects), batch_size):
                batch = all_document_objects[idx:idx+batch_size]
                print(f"    ⏳ [벡터 추가 중] 문서 {idx+1}~{idx+len(batch)}")
                vectorstore.add_documents(batch)
                gc.collect()

        except Exception as file_err:
            print(f"❌ [FATAL] 파일 처리 중 오류 발생 ({file_path}):\n{traceback.format_exc()}")

        finally:
            del all_document_objects, documents
            gc.collect()

    print("▶ [SAVE] 벡터스토어 저장 시작")
    if vectorstore:
        vectorstore.persist()
        print("✅ [DONE] 저장 완료.")
    else:
        print("⚠️ [SKIP] 저장할 벡터가 없음.")
    return vectorstore

# Example usage
if __name__ == "__main__":
    directory_path = "file/도로 교통법"
    chroma_dir = "chroma_data_folder"
    vectorstore = process_and_store_pdf_in_chroma(directory_path, chroma_dir=chroma_dir)
    retriever = vectorstore.as_retriever(search_kwargs={'k': 10})
    docs = retriever.get_relevant_documents("교통안전")
    print(f"🔎 검색 결과 개수: {len(docs)}")

In [ ]:
import re
import json
import os
import shutil
import gc
import traceback
from collections import Counter
from langchain.document_loaders import PyMuPDFLoader
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

class SentenceTransformerEmbedding:
    """
    Wrapper for SentenceTransformer to provide embed_documents and embed_query methods.
    """
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name, trust_remote_code=True)

    def embed_documents(self, texts):
        try:
            return self.model.encode(texts, batch_size=10).tolist()
        except Exception as e:
            print("❌ [임베딩 오류]:", e)
            raise

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

def clean_legal_text(text):
    text = re.sub(r'^\s*법제처.*국가법령정보센터\s*$|^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\(.*?\)\s*\d{2,3}-\d{3,4}-\d{4}', '', text)
    text = re.sub(r'(\n\s*){2,}', '\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'^\s*\[.*?\]$', '', text, flags=re.MULTILINE)
    return text.strip()

def extract_revision_dates(segment):
    revision_pattern = r"(\d{4})\.\s*(\d{1,2})\.\s*(\d{1,2})"
    matches = re.findall(revision_pattern, segment)
    return [{"year": int(year), "month": int(month), "day": int(day)} for year, month, day in matches]

def simplify_related_references(related_refs):
    simplified_refs = []
    for ref in related_refs:
        top_level = re.match(r'(제\d+조)', ref)
        if top_level:
            simplified_refs.append(top_level.group(1))
    ref_counts = Counter(simplified_refs)
    sorted_refs = sorted(ref_counts.items(), key=lambda x: (-x[1], x[0]))
    return [ref for ref, _ in sorted_refs]

def extract_metadata(segment):
    article_match = re.search(r'(제\s*\d+\s*조)\s*\((.*?)\)', segment)
    related_refs = re.findall(r'제\d+조(?:제\d+항)?(?:제\d+호)?', segment)
    simplified_refs = simplify_related_references(related_refs)
    related_refs_str = ", ".join(simplified_refs) if simplified_refs else "Not Applicable"
    revision_dates = extract_revision_dates(segment)
    revision_dates_str = json.dumps(revision_dates) if revision_dates else "Not Applicable"
    return {
        "article": article_match.group(1) if article_match else "Unknown",
        "title": article_match.group(2) if article_match else "Unknown",
        "related_references": related_refs_str,
        "revision_dates": revision_dates_str
    }

def remove_duplicates(documents):
    unique_content = {}
    unique_documents = []
    for doc in documents:
        if doc.page_content not in unique_content:
            unique_content[doc.page_content] = doc.metadata
            unique_documents.append(doc)
        else:
            unique_metadata = unique_content[doc.page_content]
            unique_metadata.update(doc.metadata)
            unique_content[doc.page_content] = unique_metadata
    return [Document(page_content=content, metadata=meta) for content, meta in unique_content.items()]

def keyword_based_splitter(text, pattern=r'제\s*\d+\s*조\s*\(.*?\)'):
    matches = re.finditer(pattern, text)
    split_positions = [match.start() for match in matches]
    split_positions.append(len(text))
    segments = []
    for i in range(len(split_positions) - 1):
        start = split_positions[i]
        end = split_positions[i + 1]
        segments.append(text[start:end].strip())
    return segments

def reset_chroma_directory(directory):
    if os.path.exists(directory):
        shutil.rmtree(directory)

# (기존 process_and_store_pdf_in_chroma 함수는 그대로 두고)  

# -------------------------
# ▶ 디버깅: Clean & Split 검증 스크립트
# -------------------------

def test_clean_and_split(file_path, page_idx=0, num_chars=300, pattern=r'제\s*\d+\s*조\s*\(.*?\)'):
    """
    1) PDF 로딩 확인
    2) 페이지별 raw 텍스트, cleaned 텍스트, split segments 확인
    """
    # 1) PDF 로드
    loader = PyMuPDFLoader(file_path)
    docs = loader.load()
    print(f"✅ 로드된 페이지 수: {len(docs)} 페이지\n")

    if len(docs) <= page_idx:
        print("지정된 페이지 인덱스가 문서 범위를 벗어났습니다.")
        return

    raw = docs[page_idx].page_content
    print(f"--- Page {page_idx+1} 원본 (앞 {num_chars}자) ---")
    print(raw[:num_chars].replace("\n","\\n"))

    # 2) 클린 텍스트
    cleaned = clean_legal_text(raw)
    print(f"\n--- Page {page_idx+1} 클린 (앞 {num_chars}자) ---")
    print(cleaned[:num_chars].replace("\n","\\n"))

    # 3) 분할
    segments = keyword_based_splitter(cleaned, pattern)
    print(f"\n--- 분할된 세그먼트: {len(segments)}개 ---")
    for idx, seg in enumerate(segments[:5], 1):
        print(f"Segment {idx} (길이 {len(seg)}자) 앞 {num_chars}자:\n{seg[:num_chars].replace('\\n','\\n')}\\n")


if __name__ == "__main__":
    test_file = "file/도로 교통법/도로교통법 시행규칙(행정안전부령)(제00507호)(20240731).pdf"
    test_clean_and_split(test_file, page_idx=0)

# 기존의 process_and_store_pdf_in_chroma 함수는 변경하지 않고,
# 위 스크립트 먼저 실행하여 clean & split 단계가 제대로 동작하는지 확인하세요.

In [ ]:
from langchain.document_loaders import PyMuPDFLoader

def load_test(file_path):
    loader = PyMuPDFLoader(file_path)
    docs = loader.load()
    print(f"✅ 페이지 수: {len(docs)}")
    for i, doc in enumerate(docs[:3], 1):
        print(f"\n--- Page {i} (문자 수: {len(doc.page_content)}) ---")
        print(doc.page_content[:200].replace("\n", "\\n"))

if __name__ == "__main__":
    load_test("file/도로 교통법/도로교통법 시행규칙(행정안전부령)(제00507호)(20240731).pdf")

In [ ]:
import re

def clean_legal_text(text):
    text = re.sub(r'^\s*법제처.*국가법령정보센터\s*$|^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\(.*?\)\s*\d{2,3}-\d{3,4}-\d{4}', '', text)
    text = re.sub(r'(\n\s*){2,}', '\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'^\s*\[.*?\]$', '', text, flags=re.MULTILINE)
    return text.strip()

def clean_test(file_path):
    loader = PyMuPDFLoader(file_path)
    raw = loader.load()[0].page_content
    cleaned = clean_legal_text(raw)
    print("--- 원본 앞 200자 ---")
    print(raw[:200].replace("\n", "\\n"))
    print("\n--- 클린 앞 200자 ---")
    print(cleaned[:200].replace("\n", "\\n"))

if __name__ == "__main__":
    clean_test("file/도로 교통법/도로교통법 시행규칙(행정안전부령)(제00507호)(20240731).pdf")

In [ ]:
import re

def keyword_based_splitter(text, pattern=r'제\s*\d+\s*조\s*\(.*?\)'):
    matches = list(re.finditer(pattern, text))
    positions = [m.start() for m in matches] + [len(text)]
    segments = []
    for i in range(len(positions)-1):
        seg = text[positions[i]:positions[i+1]].strip()
        segments.append(seg)
    return segments

def split_test(file_path):
    loader = PyMuPDFLoader(file_path)
    cleaned = clean_legal_text(loader.load()[0].page_content)
    segments = keyword_based_splitter(cleaned)
    print(f"✅ 세그먼트 개수: {len(segments)}")
    for i, seg in enumerate(segments[:5], 1):
        print(f"\n▶ Segment {i} (길이 {len(seg)}) 첫 200자:")
        print(seg[:200].replace("\n", "\\n"))

if __name__ == "__main__":
    split_test("file/도로 교통법/도로교통법 시행규칙(행정안전부령)(제00507호)(20240731).pdf")

In [ ]:
# 4) 필터링 + 출력 테스트

from langchain.document_loaders import PyMuPDFLoader

# 2~3단계 함수 재사용
def clean_legal_text(text):
    import re
    text = re.sub(r'^\s*법제처.*국가법령정보센터\s*$|^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\(.*?\)\s*\d{2,3}-\d{3,4}-\d{4}', '', text)
    text = re.sub(r'(\n\s*){2,}', '\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'^\s*\[.*?\]$', '', text, flags=re.MULTILINE)
    return text.strip()

def keyword_based_splitter(text, pattern=r'제\s*\d+\s*조\s*\(.*?\)'):
    import re
    matches = list(re.finditer(pattern, text))
    positions = [m.start() for m in matches] + [len(text)]
    segments = []
    for i in range(len(positions)-1):
        seg = text[positions[i]:positions[i+1]].strip()
        segments.append(seg)
    return segments

def filter_segments(segments, min_len=30):
    filtered = [s for s in segments if s.strip() and len(s.strip()) >= min_len]
    print(f"✅ 필터 후 세그먼트 개수: {len(filtered)}")
    return filtered

if __name__ == "__main__":
    # 1) 로드 → 클린 → 분할
    file_path = "file/도로 교통법/도로교통법 시행규칙(행정안전부령)(제00507호)(20240731).pdf"
    loader = PyMuPDFLoader(file_path)
    raw = loader.load()[0].page_content
    cleaned = clean_legal_text(raw)
    segments = keyword_based_splitter(cleaned)

    # 2) 필터링
    filtered = filter_segments(segments)

    # 3) 결과 출력 (앞 3개만)
    for idx, seg in enumerate(filtered[:3], 1):
        display = seg[:200].replace("\n", "\\n")  # 변수에 치환 결과 담기
        print(f"\n▶ Filtered Segment {idx} (길이 {len(seg)}자) 앞 200자:\n{display}")

In [ ]:
import re
import json
import os
import shutil
import gc
import traceback
from collections import Counter
from langchain.document_loaders import PyMuPDFLoader
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

class SentenceTransformerEmbedding:
    """
    Wrapper for SentenceTransformer for batch embedding.
    """
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name, trust_remote_code=True)

    def embed_documents(self, texts):
        try:
            return self.model.encode(texts, batch_size=10).tolist()
        except Exception as e:
            print("❌ [임베딩 오류]:", e)
            raise

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

def clean_legal_text(text):
    text = re.sub(r'^\s*법제처.*국가법령정보센터\s*$|^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\(.*?\)\s*\d{2,3}-\d{3,4}-\d{4}', '', text)
    text = re.sub(r'(\n\s*){2,}', '\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r'^\s*\[.*?\]$', '', text, flags=re.MULTILINE)
    return text.strip()

def keyword_based_splitter(text, pattern=r'제\s*\d+\s*조\s*\(.*?\)'):
    matches = list(re.finditer(pattern, text))
    positions = [m.start() for m in matches] + [len(text)]
    segments = []
    for i in range(len(positions) - 1):
        seg = text[positions[i]:positions[i+1]].strip()
        segments.append(seg)
    return segments

def filter_segments(segments, min_len=30):
    filtered = [s for s in segments if s.strip() and len(s.strip()) >= min_len]
    return filtered

def extract_revision_dates(segment):
    pattern = r"(\d{4})\.\s*(\d{1,2})\.\s*(\d{1,2})"
    matches = re.findall(pattern, segment)
    return [{"year": int(y), "month": int(m), "day": int(d)} for y,m,d in matches]

def simplify_related_references(refs):
    tops = [re.match(r'(제\d+조)', r).group(1) for r in refs if re.match(r'(제\d+조)', r)]
    counts = Counter(tops)
    return [ref for ref, _ in sorted(counts.items(), key=lambda x: (-x[1], x[0]))]

def extract_metadata(segment, source):
    m = re.search(r'(제\s*\d+\s*조)\s*\((.*?)\)', segment)
    refs = re.findall(r'제\d+조(?:제\d+항)?(?:제\d+호)?', segment)
    return {
        "article": m.group(1) if m else "Unknown",
        "title": m.group(2) if m else "Unknown",
        "related_references": ", ".join(simplify_related_references(refs)) or "Not Applicable",
        "revision_dates": json.dumps(extract_revision_dates(segment)) or "Not Applicable",
        "source": source
    }

def reset_chroma_dir(dir_path):
    if os.path.exists(dir_path): shutil.rmtree(dir_path)

def process_and_store_pdf_in_chroma(dir_path, chroma_dir="chroma_data", min_len=30):
    print("▶ Initializing Chroma dir")
    reset_chroma_dir(chroma_dir)
    print("▶ Initializing embedding model")
    embed_model = SentenceTransformerEmbedding("jinaai/jina-embeddings-v3")
    vectorstore = None

    for fname in os.listdir(dir_path):
        if not fname.endswith('.pdf'): continue
        path = os.path.join(dir_path, fname)
        print(f"\n📄 Processing {path}")
        pages = PyMuPDFLoader(path).load()
        all_docs = []
        for i, page in enumerate(pages):
            txt = clean_legal_text(page.page_content)
            if not txt: continue
            segs = keyword_based_splitter(txt)
            segs = filter_segments(segs, min_len)
            for seg in segs:
                meta = extract_metadata(seg, path)
                all_docs.append(Document(page_content=seg, metadata=meta))
        if not all_docs:
            print(f"⚠️ No valid segments in {fname}")
            continue
            print(f"  ✅ Segments after filter: {len(all_docs)}")
        if vectorstore is None:
            print("  ⮕ 초기 벡터스토어 생성 중")
            vectorstore = Chroma(
            persist_directory=chroma_dir,
            embedding_function=embed_model
        )

        # 3) 배치별로 문서 추가
        for i in range(0, len(all_docs), 10)(0, len(all_docs), 10):
            batch = all_docs[i:i+10]
            vectorstore.add_documents(batch)
            gc.collect()
    if vectorstore:
        print("▶ Persisting vectorstore")
        vectorstore.persist()
        print("✅ Done")
    return vectorstore

if __name__ == '__main__':
    store = process_and_store_pdf_in_chroma("file/도로 교통법", chroma_dir="chroma_data_folder")
    retr = store.as_retriever(search_kwargs={'k':10})
    docs = retr.get_relevant_documents("교통안전")
    print(f"🔎 Retrieved {len(docs)} docs")


In [ ]:
import re
import json
import os
import shutil
import gc
from collections import Counter
from langchain.document_loaders import PyMuPDFLoader
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

class SentenceTransformerEmbedding:
    """
    Wrapper for SentenceTransformer to provide embed_documents and embed_query methods.
    """
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name, trust_remote_code=True)

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

def clean_legal_text(text):
    """
    Clean legal text by removing unwanted headers, footers, and other repetitive patterns.
    """
    text = re.sub(r'^\s*법제처.*국가법령정보센터\s*$|^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\(.*?\)\s*\d{2,3}-\d{3,4}-\d{4}', '', text)
    text = re.sub(r'(\n\s*){2,}', '\n', text)  # Collapse multiple newlines
    text = re.sub(r'\s{2,}', ' ', text)        # Collapse multiple spaces
    text = re.sub(r'^\s*\[.*?\]$', '', text, flags=re.MULTILINE)
    return text.strip()

def extract_revision_dates(segment):
    """
    Extract revision dates from the text and return as a list of dictionaries with year, month, day.
    """
    revision_pattern = r"(\d{4})\.\s*(\d{1,2})\.\s*(\d{1,2})"
    matches = re.findall(revision_pattern, segment)
    return [{"year": int(year), "month": int(month), "day": int(day)} for year, month, day in matches]

def simplify_related_references(related_refs):
    """
    Simplify related references by keeping only the top-level article (e.g., '제19조'),
    and sort them by frequency (desc) and name (asc) if tied.
    """
    simplified_refs = []
    for ref in related_refs:
        top_level = re.match(r'(제\d+조)', ref)
        if top_level:
            simplified_refs.append(top_level.group(1))
    
    # Count frequencies
    ref_counts = Counter(simplified_refs)
    
    # Sort by frequency (desc), then name (asc)
    sorted_refs = sorted(ref_counts.items(), key=lambda x: (-x[1], x[0]))
    
    # Return only the sorted references
    return [ref for ref, _ in sorted_refs]

def extract_metadata(segment):
    """
    Extract metadata such as article number, title, related references, and revision dates.
    """
    article_match = re.search(r'(제\s*\d+\s*조)\s*\((.*?)\)', segment)
    related_refs = re.findall(r'제\d+조(?:제\d+항)?(?:제\d+호)?', segment)

    # Simplify and sort related references
    simplified_refs = simplify_related_references(related_refs)
    related_refs_str = ", ".join(simplified_refs) if simplified_refs else "Not Applicable"

    revision_dates = extract_revision_dates(segment)
    revision_dates_str = json.dumps(revision_dates) if revision_dates else "Not Applicable"

    metadata = {
        "article": article_match.group(1) if article_match else "Unknown",
        "title": article_match.group(2) if article_match else "Unknown",
        "related_references": related_refs_str,
        "revision_dates": revision_dates_str
    }
    return metadata

def remove_duplicates(documents):
    """
    Remove duplicate documents based on page_content.
    """
    unique_content = {}
    unique_documents = []
    for doc in documents:
        if doc.page_content not in unique_content:
            unique_content[doc.page_content] = doc.metadata
            unique_documents.append(doc)
        else:
            unique_metadata = unique_content[doc.page_content]
            unique_metadata.update(doc.metadata)
            unique_content[doc.page_content] = unique_metadata
    return [
        Document(page_content=content, metadata=meta)
        for content, meta in unique_content.items()
    ]

def keyword_based_splitter(text, pattern=r'제\s*\d+\s*조\s*\(.*?\)'):
    """
    Split text based on a keyword pattern like '제 몇 조 (블라 블라)'.
    """
    matches = re.finditer(pattern, text)
    split_positions = [match.start() for match in matches]
    split_positions.append(len(text))

    segments = []
    for i in range(len(split_positions) - 1):
        start = split_positions[i]
        end = split_positions[i + 1]
        segments.append(text[start:end].strip())
    
    return segments

def reset_chroma_directory(directory):
    """
    Delete the Chroma directory to reset the vector database.
    """
    if os.path.exists(directory):
        shutil.rmtree(directory)

def process_and_store_pdf_in_chroma(directory_path, pattern=r'제\s*\d+\s*조\s*\(.*?\)', chroma_dir="chroma_data"):
    """
    Process all PDF files in the given directory, compute embeddings, and store them in a Chroma vector database.
    """
    reset_chroma_directory(chroma_dir)

    # Initialize Chroma vectorstore and embedding model
    embedding_model = SentenceTransformerEmbedding("jinaai/jina-embeddings-v3")
    vectorstore = None

    for file_name in os.listdir(directory_path):
        if file_name.endswith(".pdf"):
            file_path = os.path.join(directory_path, file_name)
            print(f"Processing: {file_path}")

            loader = PyMuPDFLoader(file_path)
            documents = loader.load()

            # Process each document's page content separately
            all_document_objects = []  # To collect all document objects for a single file

            for doc in documents:
                # Clean and split text
                cleaned_text = clean_legal_text(doc.page_content)
                if not cleaned_text.strip():  # Skip if text is empty
                    print(f"Warning: Empty page content in {file_path}, skipping.")
                    continue

                split_segments = keyword_based_splitter(cleaned_text, pattern)

                # Create Document objects with metadata
                document_objects = []
                for segment in split_segments:
                    metadata = extract_metadata(segment)
                    metadata["source"] = file_path
                    document_objects.append(Document(page_content=segment, metadata=metadata))

                # Remove duplicates
                document_objects = remove_duplicates(document_objects)

                # Collect valid document objects
                if document_objects:
                    all_document_objects.extend(document_objects)

                # Free memory for the current page content
                del split_segments, document_objects
                gc.collect()

            # Skip adding if no valid documents for the file
            if not all_document_objects:
                print(f"Warning: No valid segments found in {file_path}, skipping.")
                continue

            # Add documents to Chroma
            if vectorstore is None:
                vectorstore = Chroma.from_documents(
                    documents=all_document_objects,
                    embedding=embedding_model,
                    persist_directory=chroma_dir
                )
            else:
                vectorstore.add_documents(all_document_objects)

            # Free memory for the current file
            del all_document_objects, documents
            gc.collect()

    # Persist vectorstore
    if vectorstore:
        vectorstore.persist()
    return vectorstore

# Example usage
directory_path = "file/도로 교통법"
chroma_dir = "test"

vectorstore = process_and_store_pdf_in_chroma(directory_path, chroma_dir=chroma_dir)
retriever = vectorstore.as_retriever(search_kwargs={'k': 10})
retriever.get_relevant_documents("교통안전")

In [ ]:
from langchain_community.vectorstores import Chroma
from sentence_transformers import SentenceTransformer

class SentenceTransformerEmbedding:
    """
    Wrapper for SentenceTransformer to provide embed_documents and embed_query methods.
    """
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name, trust_remote_code=True)

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

def load_chroma_retriever_with_embedding(chroma_dir, embedding_model_name="jinaai/jina-embeddings-v3", k=10):
    """
    Load Chroma vectorstore from a persisted directory and return a retriever with an embedding function.
    
    Args:
        chroma_dir (str): Path to the Chroma vectorstore directory.
        embedding_model_name (str): Name of the embedding model.
        k (int): Number of relevant documents to retrieve.
    
    Returns:
        retriever: Chroma retriever instance.
    """
    # Initialize the embedding modelㅋ
    embedding_model = SentenceTransformerEmbedding(embedding_model_name)
    
    # Load Chroma vectorstore with the embedding function
    vectorstore = Chroma(persist_directory=chroma_dir, embedding_function=embedding_model)
    
    # Create retriever
    retriever = vectorstore.as_retriever(search_kwargs={'k': k})
    
    return retriever

# Example usage
chroma_dir = "chroma_data_folder_final"  # Path to the Chroma directory
retriever = load_chroma_retriever_with_embedding(chroma_dir, k=10)

# Test retriever
query = "어린이 보호 구역의 제한 속도를 알려주세요"
# relevant_documents = retriever.get_relevant_documents(query)
relevant_documents = retriever.invoke(query)

# Print results
print(f"Query: {query}")
for i, doc in enumerate(relevant_documents, start=1):
    print(f"Document {i}:")
    print(f"Content: {doc.page_content[:200]}...")  # Print the first 200 characters
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

In [ ]:
from langchain_community.vectorstores import Chroma
from sentence_transformers import SentenceTransformer

class SentenceTransformerEmbedding:
    def __init__(self, model_name, device="cpu"):  # MPS/CUDA 불안하면 cpu로 고정
        self.model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
    def embed_documents(self, texts):
        # 검색용이 아니면 호출 안 됨(지금 코드는 문서 임베딩 안 함)
        return self.model.encode(texts, batch_size=32, normalize_embeddings=True).tolist()
    def embed_query(self, text):
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()

embedding_model = SentenceTransformerEmbedding("jinaai/jina-embeddings-v3", device="cpu")

vectorstore = Chroma(
    persist_directory="chroma_data_folder_final",
    embedding_function=embedding_model,
    anonymized_telemetry=False,  # 불필요한 부하/네트워크 차단
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("jinaai/jina-embeddings-v4", trust_remote_code=True)

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium."
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)
# [3, 3]

In [ ]:
python -c "import torch, torchvision; print(torch.__version__, torchvision.__version__)"

In [ ]:
import os
os.environ["CHROMA_TELEMETRY_ENABLED"] = "FALSE"

from typing import List
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
from langchain_community.vectorstores import Chroma

# --- v3 임베딩 래퍼 (LangChain 호환) ---
class SentenceTransformerEmbedding:
    def __init__(self, model_name: str, device: str = "cpu"):
        self.model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
    def embed_documents(self, texts: List[str]):
        return self.model.encode(texts, batch_size=64, normalize_embeddings=True).tolist()
    def embed_query(self, text: str):
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()

def load_chroma_retriever(chroma_dir: str, k: int = 10, device: str = "cpu"):
    embedding_fn = SentenceTransformerEmbedding("jinaai/jina-embeddings-v3", device=device)
    client_settings = Settings(anonymized_telemetry=False)
    vs = Chroma(persist_directory=chroma_dir, embedding_function=embedding_fn, client_settings=client_settings)
    return vs.as_retriever(search_kwargs={"k": k}), vs

# --- 사용 예 ---
if __name__ == "__main__":
    CHROMA_DIR = "chroma_data_folder_final_copy"  # 본인 경로
    retriever, vs = load_chroma_retriever(CHROMA_DIR, k=10, device="cpu")

    # 상태 점검
    try:
        print("docs in store:", retriever.vectorstore._collection.count())
    except Exception as e:
        print("count check warn:", e)

    # 테스트 질의
    query = "어린이 보호 구역의 제한 속도를 알려주세요"
    docs = retriever.invoke(query)
    for i, d in enumerate(docs, 1):
        print(f"\n[{i}] {d.metadata.get('source', '')}")
        print(d.page_content[:200] + ("..." if len(d.page_content) > 200 else ""))

In [ ]:
import os
os.environ["CHROMA_TELEMETRY_ENABLED"] = "FALSE"

from typing import List
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import chromadb
from langchain_community.vectorstores import Chroma

# ---- 임베딩 래퍼 ----
class SentenceTransformerEmbedding:
    def __init__(self, model_name: str, device: str = "cpu"):
        self.model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
    def embed_documents(self, texts: List[str]):
        return self.model.encode(texts, batch_size=64, normalize_embeddings=True).tolist()
    def embed_query(self, text: str):
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()

def load_chroma_retriever(chroma_dir: str, k: int = 10, device: str = "cpu"):
    embedding_fn = SentenceTransformerEmbedding("jinaai/jina-embeddings-v3", device=device)

    # ✅ 단일 클라이언트 생성 (명시적으로 "persistent" 설정)
    client_settings = Settings(
        chroma_db_impl="duckdb+parquet",
        persist_directory=chroma_dir,
        anonymized_telemetry=False,
        is_persistent=True,
    )
    client = chromadb.Client(client_settings)   # 이 클라이언트를 재사용

    # ⚠️ client_settings=... 대신 client=client를 넘깁니다.
    vs = Chroma(
        client=client,
        collection_name="langchain",           # 기존 컬렉션명이 다르면 맞춰주세요
        embedding_function=embedding_fn,
        persist_directory=chroma_dir,
    )
    retriever = vs.as_retriever(search_kwargs={"k": k})
    return retriever, vs

# --- 사용 예 ---
if __name__ == "__main__":
    CHROMA_DIR = "chroma_data_folder_final_copy"
    retriever, vs = load_chroma_retriever(CHROMA_DIR, k=10, device="cpu")

    print("docs in store:", retriever.vectorstore._collection.count())
    q = "어린이 보호 구역의 제한 속도를 알려주세요"
    docs = retriever.invoke(q)
    for i, d in enumerate(docs, 1):
        print(f"\n[{i}] {d.metadata.get('source', '')}")
        print(d.page_content[:200] + ("..." if len(d.page_content) > 200 else ""))

In [ ]:
# ===========================
# 법령 PDF 전처리 + "제 N조(…)" 헤더 기준 스플리팅
# - 참조 표현(예: "제 1조의 내용")에서는 절대 분할 X
# - 본문 중간에 붙은 "제 N조(…)"는 헤더로 인식해 줄바꿈 1회만 강제
# - 제목 라인/헤더푸터/연락처/개정메타 제거 로직 유지
# ===========================
import os
import re
from glob import glob
from typing import List
from tqdm import tqdm

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

# ---------- 사용자 설정 ----------
PDF_DIR = "file/도로 교통법"   # 처리할 PDF 폴더
MAX_FILES = None               # None=전체, 숫자=일부
PRINT_SAMPLE_DOCS = 40         # 샘플 개수
SNIPPET_CHARS = 400            # 샘플 글자수

# ---------- 파일명 끝 (YYYYMMDD) 제거 ----------
TITLE_DATE_SUFFIX_RE = re.compile(r"\(\d{8}\)$")
def normalize_title_from_source(src: str) -> str:
    base = os.path.basename(src)
    name, _ = os.path.splitext(base)
    return TITLE_DATE_SUFFIX_RE.sub("", name).strip()

# ---------- 제목/별칭 생성 & 비교 유틸 ----------
NON_WORDS_RE = re.compile(r"[ \t\r\n0-9\-\.\,\:\;\(\)\[\]<>_/|~`'\"“”‘’·ㆍ]+")

def strip_parens(s: str) -> str:
    # 괄호류 안의 부가 정보를 제거
    return re.sub(r"\s*[$begin:math:text$\\[].*?[$end:math:text$\]]", "", s).strip()

def norm_comp_key(s: str) -> str:
    # 비교용 정규화: 공백/숫자/구두점 제거
    return NON_WORDS_RE.sub("", s)

def build_title_aliases(law_title: str):
    base = law_title.strip()
    no_paren = strip_parens(base)  # 예: "도로교통법 시행규칙"

    # 공백 유연 정규식(라인 일치용)
    def flex_spaces(text: str) -> re.Pattern:
        pat = re.escape(text.strip())
        pat = re.sub(r"\s+", r"\\s*", pat)
        return re.compile(r"^\s*" + pat + r"\s*$")

    aliases = {
        "raw": base,
        "no_paren": no_paren,
        "raw_norm": norm_comp_key(base),
        "no_paren_norm": norm_comp_key(no_paren),
        "raw_re": flex_spaces(base),
        "no_paren_re": flex_spaces(no_paren),
    }
    return aliases

def is_title_line(line: str, aliases: dict) -> bool:
    # (1) 정확/유사(공백 유연) 라인 일치
    if aliases["raw_re"].match(line) or aliases["no_paren_re"].match(line):
        return True
    # (2) 정규화 키로 완전 동일 (공백/숫자/구두점 제거 후 동일)
    nline = norm_comp_key(line)
    if nline and (nline == aliases["raw_norm"] or nline == aliases["no_paren_norm"]):
        return True
    # (3) 짧은 코어 제목이 한 줄로만 있는 경우
    core = aliases["no_paren_norm"]
    if len(core) >= 6 and nline == core:
        return True
    return False

# ---------- 제거/정규화 규칙 ----------
TOP_BRACKET_BLOCK_RE = re.compile(r"^\s*(?:$begin:math:display$[^$end:math:display$\n]+\]\s*){1,20}", flags=re.MULTILINE)

HEADER_FOOTER_RE = re.compile(
    "|".join([
        r"^\s*법제처.*국가법령정보센터.*$",
        r"^\s*국가법령정보센터.*$",
        r"^\s*www\.law\.go\.kr\s*$",
        r"^\s*페이지\s*\d+\s*$",
        r"^\s*[-–—―]+\s*$",
        r"^\s*\d+\s*$",
    ]),
    flags=re.IGNORECASE | re.MULTILINE,
)

PHONE_RE = re.compile(r"\b\d{2,3}-\d{3,4}-\d{4}\b")
ORG_HINT_RE = re.compile(r"(경찰청|행정안전부|법제처|담당부서|문의|전화|Fax|팩스|연락처)", re.IGNORECASE)
def is_contact_line(line: str) -> bool:
    return bool(PHONE_RE.search(line) and ORG_HINT_RE.search(line))

# 각/대괄호 메타(어디에 있어도 제거) — 개정/신설/삭제 등 표지만 제거
META_KEYWORDS = r"(개정|전문개정|전부개정|일부개정|신설|본조신설|삭제|정정|준용|부칙|제목개정|이동|전단|후단|종전|현행)"
ANGLE_META_ANY_RE = re.compile(r"\s*<(?=[^>]*" + META_KEYWORDS + r")[^>]*>\s*")
BRACKET_META_ANY_RE = re.compile(r"\s*$begin:math:display$(?=[^$end:math:display$]*" + META_KEYWORDS + r")[^\]]*\]\s*")
BROKEN_BRACKET_META_RE = re.compile(r"$begin:math:display$\\s*(?=[^$end:math:display$\n]*" + META_KEYWORDS + r")[^\]\n]*")

# ---------- 핵심 패턴: "제 N조(" 형태의 '정식 조문 헤더'만 ----------
# - '제 1조의 …'는 (?!\s*의)로 배제
# - 반드시 '('가 바로 이어지는 헤더만 허용
ARTICLE_HEADER_INLINE = r"(제\s*\d+\s*조)(?!\s*의)\s*\("   # 인라인/라인 공통 탐지용
ARTICLE_HEADER_INLINE_RE = re.compile(ARTICLE_HEADER_INLINE)
ARTICLE_HEADER_LINE_RE   = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

# 본문 중간의 "제목 + 조 헤더" 패턴 제거 (제목만 제거)
def remove_inline_title_before_article(txt: str, law_title: str):
    core_title = strip_parens(law_title)
    title_inline_re = re.compile(r"(?m)(^|\n)\s*" + re.escape(core_title) + rf"\s+(?={ARTICLE_HEADER_INLINE})")
    return title_inline_re.sub("\n\n", txt)

# 본문 어디에 있어도 "제목 한 줄"이 단독으로 낀 경우 제거
def remove_lonely_title_lines(txt: str, aliases: dict):
    lines = txt.splitlines()
    kept = []
    for raw in lines:
        line = raw.strip()
        if not line:
            kept.append(raw)
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(raw)
    return "\n".join(kept)

# ---------- 본문 클리너 ----------
def clean_text_legal(text: str, law_title: str) -> str:
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # 상단 메타/헤더푸터 제거
    text = TOP_BRACKET_BLOCK_RE.sub("", text, count=1)
    text = HEADER_FOOTER_RE.sub("", text)

    aliases = build_title_aliases(law_title)

    # 줄 단위 기본 필터링(연락처/기관 안내/제목 라인)
    kept = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            kept.append("")
            continue
        if is_contact_line(line):
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(line)
    txt = "\n".join(kept)

    # 본문 중간 제목+조헤더 패턴 정리
    txt = remove_inline_title_before_article(txt, law_title)

    # 개정/신설 표지 제거 ([..], <..>)
    txt = ANGLE_META_ANY_RE.sub(" ", txt)
    txt = BRACKET_META_ANY_RE.sub(" ", txt)
    txt = BROKEN_BRACKET_META_RE.sub(" ", txt)

    # 혹시 남은 단독 제목 라인 한 번 더 제거
    txt = remove_lonely_title_lines(txt, aliases)

    # ====== 조 헤더 앞/주변 개행 정규화 (엄격 헤더에만 적용) ======
    # 0) 본문 중간에 붙은 "… 제 N조(" 앞에 줄바꿈 1회 강제
    txt = re.sub(rf"(?!^)(?<!\n){ARTICLE_HEADER_INLINE}", r"\n\1(", txt)

    # 1) 조 헤더 앞 여러 빈 줄 → 정확히 1줄
    txt = re.sub(rf"(?m)\n{{2,}}(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)

    # 2) 문서 시작에서 곧바로 헤더가 오면 앞에 1줄 추가(가독)
    txt = re.sub(rf"(?m)\A(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)

    # 하이픈 줄바꿈 제거 + 공백/개행 정리
    txt = re.sub(r"-\n", "", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    # 일반 빈 줄은 최대 1줄
    txt = re.sub(r"\n{3,}", "\n\n", txt)

    return txt.strip()

# ---------- 스플리팅: "제 N조(" 헤더 라인만 ----------
ARTICLE_SPLIT_RE = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

def split_into_articles_only_jo(text: str, law_title: str, source_meta: dict) -> List[Document]:
    matches = list(ARTICLE_SPLIT_RE.finditer(text))
    if not matches:
        return [Document(page_content=f"{law_title}\n{text}", metadata=source_meta)]

    articles: List[Document] = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if not chunk:
            continue
        # 조의 n(예: 제2조의2, 제14조의3 등)은 같은 블록 내부에 그대로 유지됨
        chunk = f"{law_title} {chunk}"
        # 블록 내부 여분 개행 정리
        chunk = re.sub(r"\n{3,}", "\n\n", chunk)
        articles.append(Document(page_content=chunk, metadata=source_meta))
    return articles

# ---------- 메인 ----------
if __name__ == "__main__":
    pdf_paths = sorted(glob(os.path.join(PDF_DIR, "*.pdf")))
    if MAX_FILES:
        pdf_paths = pdf_paths[:MAX_FILES]
    if not pdf_paths:
        raise FileNotFoundError(f"'{PDF_DIR}' 폴더에 PDF가 없습니다.")

    all_articles: List[Document] = []

    for p in tqdm(pdf_paths, desc="Processing PDFs"):
        loader = PyMuPDFLoader(p)
        pages = loader.load()

        law_title = normalize_title_from_source(p)
        page_texts = [clean_text_legal(d.page_content or "", law_title) for d in pages]
        # 페이지 합칠 때는 페이지 사이 1개 빈 줄만
        merged = "\n".join([t for t in page_texts if t.strip()])

        docs = split_into_articles_only_jo(merged, law_title, {"source": p, "law_title": law_title})
        all_articles.extend(docs)

    print(f"[INFO] 전처리/분할 완료. 조문 단위 문서 수: {len(all_articles)}")

    # 샘플 출력
    for i, d in enumerate(all_articles[:PRINT_SAMPLE_DOCS], start=1):
        print(f"\n--- 조문 샘플 {i} ---")
        print("law_title:", d.metadata.get("law_title"))
        print(d.page_content)#[:SNIPPET_CHARS])

In [ ]:
 # ===========================
# 법령 PDF 전처리 + "제 N조(…)" 헤더 기준 스플리팅
# - 참조 표현(예: "제 1조의 내용")에서는 절대 분할 X
# - 본문 중간에 붙은 "제 N조(…)"는 헤더로 인식해 줄바꿈 1회만 강제
# - []·<> 메타(개정/신설/이동/부칙 등) 전부 제거(멀티라인 포함)
# - 단, '삭제<YYYY. M. D.>' 형식은 반드시 보존
# ===========================
import os
import re
from glob import glob
from typing import List
from tqdm import tqdm

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

# ---------- 사용자 설정 ----------
PDF_DIR = "file/도로 교통법"   # 처리할 PDF 폴더
MAX_FILES = None               # None=전체, 숫자=일부
PRINT_SAMPLE_DOCS = 10         # 샘플 개수
SNIPPET_CHARS = 400            # 샘플 글자수

# ---------- 파일명 끝 (YYYYMMDD) 제거 ----------
TITLE_DATE_SUFFIX_RE = re.compile(r"\(\d{8}\)$")
def normalize_title_from_source(src: str) -> str:
    base = os.path.basename(src)
    name, _ = os.path.splitext(base)
    return TITLE_DATE_SUFFIX_RE.sub("", name).strip()

# ---------- 제목/별칭 생성 & 비교 유틸 ----------
NON_WORDS_RE = re.compile(r"[ \t\r\n0-9\-\.\,\:\;\(\)\[\]<>_/|~`'\"“”‘’·ㆍ]+")

def strip_parens(s: str) -> str:
    # 괄호류 안의 부가 정보를 제거
    return re.sub(r"\s*[$begin:math:text$\\[].*?[$end:math:text$\]]", "", s).strip()

def norm_comp_key(s: str) -> str:
    # 비교용 정규화: 공백/숫자/구두점 제거
    return NON_WORDS_RE.sub("", s)

def build_title_aliases(law_title: str):
    base = law_title.strip()
    no_paren = strip_parens(base)  # 예: "도로교통법 시행규칙"

    # 공백 유연 정규식(라인 일치용)
    def flex_spaces(text: str) -> re.Pattern:
        pat = re.escape(text.strip())
        pat = re.sub(r"\s+", r"\\s*", pat)
        return re.compile(r"^\s*" + pat + r"\s*$")

    aliases = {
        "raw": base,
        "no_paren": no_paren,
        "raw_norm": norm_comp_key(base),
        "no_paren_norm": norm_comp_key(no_paren),
        "raw_re": flex_spaces(base),
        "no_paren_re": flex_spaces(no_paren),
    }
    return aliases

def is_title_line(line: str, aliases: dict) -> bool:
    # (1) 정확/유사(공백 유연) 라인 일치
    if aliases["raw_re"].match(line) or aliases["no_paren_re"].match(line):
        return True
    # (2) 정규화 키로 완전 동일 (공백/숫자/구두점 제거 후 동일)
    nline = norm_comp_key(line)
    if nline and (nline == aliases["raw_norm"] or nline == aliases["no_paren_norm"]):
        return True
    # (3) 짧은 코어 제목이 한 줄로만 있는 경우
    core = aliases["no_paren_norm"]
    if len(core) >= 6 and nline == core:
        return True
    return False

# ---------- 제거/정규화 규칙 ----------
TOP_BRACKET_BLOCK_RE = re.compile(r"^\s*(?:$begin:math:display$[^$end:math:display$\n]+\]\s*){1,20}", flags=re.MULTILINE)

HEADER_FOOTER_RE = re.compile(
    "|".join([
        r"^\s*법제처.*국가법령정보센터.*$",
        r"^\s*국가법령정보센터.*$",
        r"^\s*www\.law\.go\.kr\s*$",
        r"^\s*페이지\s*\d+\s*$",
        r"^\s*[-–—―]+\s*$",
        r"^\s*\d+\s*$",
    ]),
    flags=re.IGNORECASE | re.MULTILINE,
)

PHONE_RE = re.compile(r"\b\d{2,3}-\d{3,4}-\d{4}\b")
ORG_HINT_RE = re.compile(r"(경찰청|행정안전부|법제처|담당부서|문의|전화|Fax|팩스|연락처)", re.IGNORECASE)
def is_contact_line(line: str) -> bool:
    return bool(PHONE_RE.search(line) and ORG_HINT_RE.search(line))

# --- 멀티라인까지 제거하는 메타 패턴 (실제 []/<> 기준) ---
META_KEYWORDS = r"(개정|전문개정|전부개정|일부개정|신설|본조신설|삭제|정정|준용|부칙|제목개정|이동|전단|후단|종전|현행)"
# 대괄호 전체 제거: [ ... META_KEYWORDS ... ]
BRACKET_META_ANY_RE = re.compile(r"\s*\[(?=[\s\S]*?" + META_KEYWORDS + r")[\s\S]*?\]\s*")
# 꺾쇠 전체 제거: < ... META_KEYWORDS ... >
ANGLE_META_ANY_RE   = re.compile(r"\s*<(?=[\s\S]*?" + META_KEYWORDS + r")[\s\S]*?>\s*")
# 깨진 대괄호 조각 방지(옵션)
BROKEN_BRACKET_META_RE = re.compile(r"$begin:math:display$\\s*(?=[\\s\\S]*?" + META_KEYWORDS + r")[^$end:math:display$]*")

# ---------- 핵심 패턴: "제 N조(" 형태의 '정식 조문 헤더'만 ----------
# - '제 1조의 …'는 (?!\s*의)로 배제
# - 반드시 '('가 바로 이어지는 헤더만 허용
ARTICLE_HEADER_INLINE = r"(제\s*\d+\s*조)(?!\s*의)\s*\("   # 인라인/라인 공통 탐지용
ARTICLE_HEADER_INLINE_RE = re.compile(ARTICLE_HEADER_INLINE)
ARTICLE_HEADER_LINE_RE   = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

# 본문 중간의 "제목 + 조 헤더" 패턴 제거 (제목만 제거)
def remove_inline_title_before_article(txt: str, law_title: str):
    core_title = strip_parens(law_title)
    title_inline_re = re.compile(r"(?m)(^|\n)\s*" + re.escape(core_title) + rf"\s+(?={ARTICLE_HEADER_INLINE})")
    return title_inline_re.sub("\n\n", txt)

# 본문 어디에 있어도 "제목 한 줄"이 단독으로 낀 경우 제거
def remove_lonely_title_lines(txt: str, aliases: dict):
    lines = txt.splitlines()
    kept = []
    for raw in lines:
        line = raw.strip()
        if not line:
            kept.append(raw)
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(raw)
    return "\n".join(kept)

# ---------- 본문 클리너 ----------
def clean_text_legal(text: str, law_title: str) -> str:
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # 상단 메타/헤더푸터 제거
    text = TOP_BRACKET_BLOCK_RE.sub("", text, count=1)
    text = HEADER_FOOTER_RE.sub("", text)

    aliases = build_title_aliases(law_title)

    # 줄 단위 기본 필터링(연락처/기관 안내/제목 라인)
    kept = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            kept.append("")
            continue
        if is_contact_line(line):
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(line)
    txt = "\n".join(kept)

    # 본문 중간 제목+조헤더 패턴 정리
    txt = remove_inline_title_before_article(txt, law_title)

    # --- (보호) '삭제<...>'는 남겨야 하므로 각도괄호를 임시 토큰으로 보호 ---
    def _protect_delete_angle(m):
        return m.group(0).replace("<", "«KEEP_ANGLE_OPEN»").replace(">", "«KEEP_ANGLE_CLOSE»")
    # '삭제<' 또는 '전부 삭제<' 변형까지 보호 (멀티라인)
    txt = re.sub(r"(?:전부\s*)?삭제\s*<[\s\S]*?>", _protect_delete_angle, txt)

    # --- 메타 제거: []·<> 안의 META_KEYWORDS 포함 블록은 전부 삭제 (멀티라인) ---
    txt = BRACKET_META_ANY_RE.sub(" ", txt)     # [본조신설 …], [제37조의2에서 이동 <…>] 등
    txt = ANGLE_META_ANY_RE.sub(" ", txt)       # <개정 …>, <… 이동 …> 등
    txt = BROKEN_BRACKET_META_RE.sub(" ", txt)  # 깨진 대괄호 조각도 정리

    # --- 보호 토큰 원복 ---
    txt = txt.replace("«KEEP_ANGLE_OPEN»", "<").replace("«KEEP_ANGLE_CLOSE»", ">")

    # 혹시 남은 단독 제목 라인 한 번 더 제거
    txt = remove_lonely_title_lines(txt, aliases)

    # ====== 조 헤더 앞/주변 개행 정규화 (엄격 헤더에만 적용) ======
    # 0) 본문 중간에 붙은 "… 제 N조(" 앞에 줄바꿈 1회 강제
    txt = re.sub(rf"(?!^)(?<!\n){ARTICLE_HEADER_INLINE}", r"\n\1(", txt)

    # 1) 조 헤더 앞 여러 빈 줄 → 정확히 1줄
    txt = re.sub(rf"(?m)\n{{2,}}(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)

    # 2) 문서 시작에서 곧바로 헤더가 오면 앞에 1줄 추가(가독)
    txt = re.sub(rf"(?m)\A(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)

    # 하이픈 줄바꿈 제거 + 공백/개행 정리
    txt = re.sub(r"-\n", "", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    # 일반 빈 줄은 최대 1줄
    txt = re.sub(r"\n{3,}", "\n\n", txt)

    return txt.strip()

# ---------- 스플리팅: "제 N조(" 헤더 라인만 ----------
ARTICLE_SPLIT_RE = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

def split_into_articles_only_jo(text: str, law_title: str, source_meta: dict) -> List[Document]:
    matches = list(ARTICLE_SPLIT_RE.finditer(text))
    if not matches:
        return [Document(page_content=f"{law_title}\n{text}", metadata=source_meta)]

    articles: List[Document] = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if not chunk:
            continue
        # 조의 n(예: 제2조의2, 제14조의3 등)은 같은 블록 내부에 그대로 유지됨
        chunk = f"{law_title} {chunk}"
        # 블록 내부 여분 개행 정리
        chunk = re.sub(r"\n{3,}", "\n\n", chunk)
        articles.append(Document(page_content=chunk, metadata=source_meta))
    return articles

# ---------- 메인 ----------
if __name__ == "__main__":
    pdf_paths = sorted(glob(os.path.join(PDF_DIR, "*.pdf")))
    if MAX_FILES:
        pdf_paths = pdf_paths[:MAX_FILES]
    if not pdf_paths:
        raise FileNotFoundError(f"'{PDF_DIR}' 폴더에 PDF가 없습니다.")

    all_articles: List[Document] = []

    for p in tqdm(pdf_paths, desc="Processing PDFs"):
        loader = PyMuPDFLoader(p)
        pages = loader.load()

        law_title = normalize_title_from_source(p)
        page_texts = [clean_text_legal(d.page_content or "", law_title) for d in pages]
        # 페이지 합칠 때는 페이지 사이 1개 빈 줄만
        merged = "\n".join([t for t in page_texts if t.strip()])

        docs = split_into_articles_only_jo(merged, law_title, {"source": p, "law_title": law_title})
        all_articles.extend(docs)

    print(f"[INFO] 전처리/분할 완료. 조문 단위 문서 수: {len(all_articles)}")

    # 샘플 출력
    for i, d in enumerate(all_articles[:PRINT_SAMPLE_DOCS], start=1):
        print(f"\n--- 조문 샘플 {i} ---")
        print("law_title:", d.metadata.get("law_title"))
        print(d.page_content)#[:SNIPPET_CHARS])

In [ ]:
 # ===========================
# 법령 PDF 전처리 + "제 N조(…)" 헤더 기준 스플리팅
# - 참조 표현(예: "제 1조의 내용")에서는 절대 분할 X
# - 본문 중간에 붙은 "제 N조(…)"는 헤더로 인식해 줄바꿈 1회만 강제
# - []·<> 메타(개정/신설/이동/부칙 등) 전부 제거(멀티라인 포함)
# - 단, '삭제<YYYY. M. D.>' 형식은 반드시 보존
# ===========================
import os
import re
from glob import glob
from typing import List
from tqdm import tqdm

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

# ---------- 사용자 설정 ----------
PDF_DIR = "file/도로 교통법"   # 처리할 PDF 폴더
MAX_FILES = None               # None=전체, 숫자=일부
PRINT_SAMPLE_DOCS = 10         # 샘플 개수
SNIPPET_CHARS = 400            # 샘플 글자수

# ---------- 파일명 끝 (YYYYMMDD) 제거 ----------
TITLE_DATE_SUFFIX_RE = re.compile(r"\(\d{8}\)$")
def normalize_title_from_source(src: str) -> str:
    base = os.path.basename(src)
    name, _ = os.path.splitext(base)
    return TITLE_DATE_SUFFIX_RE.sub("", name).strip()

# ---------- 제목/별칭 생성 & 비교 유틸 ----------
NON_WORDS_RE = re.compile(r"[ \t\r\n0-9\-\.\,\:\;\(\)\[\]<>_/|~`'\"“”‘’·ㆍ]+")

def strip_parens(s: str) -> str:
    # 괄호류 안의 부가 정보를 제거
    return re.sub(r"\s*[$begin:math:text$\\[].*?[$end:math:text$\]]", "", s).strip()

def norm_comp_key(s: str) -> str:
    # 비교용 정규화: 공백/숫자/구두점 제거
    return NON_WORDS_RE.sub("", s)

def build_title_aliases(law_title: str):
    base = law_title.strip()
    no_paren = strip_parens(base)  # 예: "도로교통법 시행규칙"

    # 공백 유연 정규식(라인 일치용)
    def flex_spaces(text: str) -> re.Pattern:
        pat = re.escape(text.strip())
        pat = re.sub(r"\s+", r"\\s*", pat)
        return re.compile(r"^\s*" + pat + r"\s*$")

    aliases = {
        "raw": base,
        "no_paren": no_paren,
        "raw_norm": norm_comp_key(base),
        "no_paren_norm": norm_comp_key(no_paren),
        "raw_re": flex_spaces(base),
        "no_paren_re": flex_spaces(no_paren),
    }
    return aliases

def is_title_line(line: str, aliases: dict) -> bool:
    # (1) 정확/유사(공백 유연) 라인 일치
    if aliases["raw_re"].match(line) or aliases["no_paren_re"].match(line):
        return True
    # (2) 정규화 키로 완전 동일 (공백/숫자/구두점 제거 후 동일)
    nline = norm_comp_key(line)
    if nline and (nline == aliases["raw_norm"] or nline == aliases["no_paren_norm"]):
        return True
    # (3) 짧은 코어 제목이 한 줄로만 있는 경우
    core = aliases["no_paren_norm"]
    if len(core) >= 6 and nline == core:
        return True
    return False

# ---------- 제거/정규화 규칙 ----------
TOP_BRACKET_BLOCK_RE = re.compile(r"^\s*(?:$begin:math:display$[^$end:math:display$\n]+\]\s*){1,20}", flags=re.MULTILINE)


HEADER_FOOTER_RE = re.compile(
    "|".join([
        r"^\s*법제처.*국가법령정보센터.*$",
        r"^\s*국가법령정보센터.*$",
        r"^\s*www\.law\.go\.kr\s*$",
        r"^\s*페이지\s*\d+\s*$",
        r"^\s*[-–—―]+\s*$",
        r"^\s*\d+\s*$",
    ]),
    flags=re.IGNORECASE | re.MULTILINE,
)

PHONE_RE = re.compile(r"\b\d{2,3}-\d{3,4}-\d{4}\b")
ORG_HINT_RE = re.compile(r"(경찰청|행정안전부|법제처|담당부서|문의|전화|Fax|팩스|연락처)", re.IGNORECASE)
def is_contact_line(line: str) -> bool:
    return bool(PHONE_RE.search(line) and ORG_HINT_RE.search(line))

# --- 멀티라인까지 제거하는 메타 패턴 (실제 []/<> 기준) ---
META_KEYWORDS = r"(개정|전문개정|전부개정|일부개정|신설|본조신설|삭제|정정|준용|부칙|제목개정|이동|전단|후단|종전|현행)"
# 대괄호 전체 제거: [ ... META_KEYWORDS ... ]
BRACKET_META_ANY_RE = re.compile(r"\s*\[(?=[\s\S]*?" + META_KEYWORDS + r")[\s\S]*?\]\s*")
# 꺾쇠 전체 제거: < ... META_KEYWORDS ... >
ANGLE_META_ANY_RE   = re.compile(r"\s*<(?=[\s\S]*?" + META_KEYWORDS + r")[\s\S]*?>\s*")
# 깨진 대괄호 조각 방지(옵션)
BROKEN_BRACKET_META_RE = re.compile(r"$begin:math:display$\\s*(?=[\\s\\S]*?" + META_KEYWORDS + r")[^$end:math:display$]*")
# --- 장(章) 헤더 라인 제거 ---
# 예: "제2장 보행자의 통행방법" 같은 "한 줄" 전체 삭제
CHAPTER_HEADER_LINE_RE = re.compile(r"(?m)^\s*제\s*\d+\s*장(?:\s+[^\n]*)?\s*$")

# ---------- 핵심 패턴: "제 N조(" 형태의 '정식 조문 헤더'만 ----------
# - '제 1조의 …'는 (?!\s*의)로 배제
# - 반드시 '('가 바로 이어지는 헤더만 허용
ARTICLE_HEADER_INLINE = r"(제\s*\d+\s*조)(?!\s*의)\s*\("   # 인라인/라인 공통 탐지용
ARTICLE_HEADER_INLINE_RE = re.compile(ARTICLE_HEADER_INLINE)
ARTICLE_HEADER_LINE_RE   = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

# 본문 중간의 "제목 + 조 헤더" 패턴 제거 (제목만 제거)
def remove_inline_title_before_article(txt: str, law_title: str):
    core_title = strip_parens(law_title)
    title_inline_re = re.compile(r"(?m)(^|\n)\s*" + re.escape(core_title) + rf"\s+(?={ARTICLE_HEADER_INLINE})")
    return title_inline_re.sub("\n\n", txt)

# 본문 어디에 있어도 "제목 한 줄"이 단독으로 낀 경우 제거
def remove_lonely_title_lines(txt: str, aliases: dict):
    lines = txt.splitlines()
    kept = []
    for raw in lines:
        line = raw.strip()
        if not line:
            kept.append(raw)
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(raw)
    return "\n".join(kept)

# ---------- 본문 클리너 ----------
def clean_text_legal(text: str, law_title: str) -> str:
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # 상단 메타/헤더푸터 제거
    text = TOP_BRACKET_BLOCK_RE.sub("", text, count=1)
    text = HEADER_FOOTER_RE.sub("", text)

    aliases = build_title_aliases(law_title)

    # 줄 단위 기본 필터링(연락처/기관 안내/제목 라인)
    kept = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            kept.append("")
            continue
        if is_contact_line(line):
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(line)
    txt = "\n".join(kept)

    # 본문 중간 제목+조헤더 패턴 정리
    txt = remove_inline_title_before_article(txt, law_title)

    # --- (보호) '삭제<...>'는 남겨야 하므로 각도괄호를 임시 토큰으로 보호 ---
    def _protect_delete_angle(m):
        return m.group(0).replace("<", "«KEEP_ANGLE_OPEN»").replace(">", "«KEEP_ANGLE_CLOSE»")
    # '삭제<' 또는 '전부 삭제<' 변형까지 보호 (멀티라인)
    txt = re.sub(r"(?:전부\s*)?삭제\s*<[\s\S]*?>", _protect_delete_angle, txt)

    # --- 메타 제거: []·<> 안의 META_KEYWORDS 포함 블록은 전부 삭제 (멀티라인) ---
    txt = BRACKET_META_ANY_RE.sub(" ", txt)     # [본조신설 …], [제37조의2에서 이동 <…>] 등
    txt = ANGLE_META_ANY_RE.sub(" ", txt)       # <개정 …>, <… 이동 …> 등
    txt = BROKEN_BRACKET_META_RE.sub(" ", txt)  # 깨진 대괄호 조각도 정리

    # --- 보호 토큰 원복 ---
    txt = txt.replace("«KEEP_ANGLE_OPEN»", "<").replace("«KEEP_ANGLE_CLOSE»", ">")

    # 혹시 남은 단독 제목 라인 한 번 더 제거
    txt = remove_lonely_title_lines(txt, aliases)

    # ====== 조 헤더 앞/주변 개행 정규화 (엄격 헤더에만 적용) ======
    # 0) 본문 중간에 붙은 "… 제 N조(" 앞에 줄바꿈 1회 강제
    txt = re.sub(rf"(?!^)(?<!\n){ARTICLE_HEADER_INLINE}", r"\n\1(", txt)

    # 1) 조 헤더 앞 여러 빈 줄 → 정확히 1줄
    txt = re.sub(rf"(?m)\n{{2,}}(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)

    # 2) 문서 시작에서 곧바로 헤더가 오면 앞에 1줄 추가(가독)
    txt = re.sub(rf"(?m)\A(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)
    # --- 보호 토큰 원복 ---
    txt = txt.replace("«KEEP_ANGLE_OPEN»", "<").replace("«KEEP_ANGLE_CLOSE»", ">")

    # >>> 여기 한 줄 추가
    txt = CHAPTER_HEADER_LINE_RE.sub("", txt)

    # 혹시 남은 단독 제목 라인 한 번 더 제거
    txt = remove_lonely_title_lines(txt, aliases)

    # 하이픈 줄바꿈 제거 + 공백/개행 정리
    txt = re.sub(r"-\n", "", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    # 일반 빈 줄은 최대 1줄
    txt = re.sub(r"\n{3,}", "\n\n", txt)

    return txt.strip()

# ---------- 스플리팅: "제 N조(" 헤더 라인만 ----------
ARTICLE_SPLIT_RE = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

def split_into_articles_only_jo(text: str, law_title: str, source_meta: dict) -> List[Document]:
    matches = list(ARTICLE_SPLIT_RE.finditer(text))
    if not matches:
        return [Document(page_content=f"{law_title}\n{text}", metadata=source_meta)]

    articles: List[Document] = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if not chunk:
            continue
        # 조의 n(예: 제2조의2, 제14조의3 등)은 같은 블록 내부에 그대로 유지됨
        chunk = f"{law_title} {chunk}"
        # 블록 내부 여분 개행 정리
        chunk = re.sub(r"\n{3,}", "\n\n", chunk)
        articles.append(Document(page_content=chunk, metadata=source_meta))
    return articles

# ---------- 메인 ----------
if __name__ == "__main__":
    pdf_paths = sorted(glob(os.path.join(PDF_DIR, "*.pdf")))
    if MAX_FILES:
        pdf_paths = pdf_paths[:MAX_FILES]
    if not pdf_paths:
        raise FileNotFoundError(f"'{PDF_DIR}' 폴더에 PDF가 없습니다.")

    all_articles: List[Document] = []

    for p in tqdm(pdf_paths, desc="Processing PDFs"):
        loader = PyMuPDFLoader(p)
        pages = loader.load()

        law_title = normalize_title_from_source(p)
        page_texts = [clean_text_legal(d.page_content or "", law_title) for d in pages]
        # 페이지 합칠 때는 페이지 사이 1개 빈 줄만
        merged = "\n".join([t for t in page_texts if t.strip()])

        docs = split_into_articles_only_jo(merged, law_title, {"source": p, "law_title": law_title})
        all_articles.extend(docs)

    print(f"[INFO] 전처리/분할 완료. 조문 단위 문서 수: {len(all_articles)}")

    # 샘플 출력
    for i, d in enumerate(all_articles[:PRINT_SAMPLE_DOCS], start=1):
        print(f"\n--- 조문 샘플 {i} ---")
        print("law_title:", d.metadata.get("law_title"))
        print(d.page_content)#[:SNIPPET_CHARS])

In [ ]:
 # ===========================
# 법령 PDF 전처리 + "제 N조(…)" 헤더 기준 스플리팅
# - 참조 표현(예: "제 1조의 내용")에서는 절대 분할 X
# - 본문 중간에 붙은 "제 N조(…)"는 헤더로 인식해 줄바꿈 1회만 강제
# - []·<> 메타(개정/신설/이동/부칙 등) 전부 제거(멀티라인 포함)
# - 단, '삭제<YYYY. M. D.>' 형식은 반드시 보존
# ===========================
import os
import re
from glob import glob
from typing import List
from tqdm import tqdm

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

# ---------- 사용자 설정 ----------
PDF_DIR = "file/도로 교통법"   # 처리할 PDF 폴더
MAX_FILES = None               # None=전체, 숫자=일부
PRINT_SAMPLE_DOCS = 40         # 샘플 개수
SNIPPET_CHARS = 400            # 샘플 글자수

# ---------- 파일명 끝 (YYYYMMDD) 제거 ----------
TITLE_DATE_SUFFIX_RE = re.compile(r"\(\d{8}\)$")
def normalize_title_from_source(src: str) -> str:
    base = os.path.basename(src)
    name, _ = os.path.splitext(base)
    return TITLE_DATE_SUFFIX_RE.sub("", name).strip()

# ---------- 제목/별칭 생성 & 비교 유틸 ----------
NON_WORDS_RE = re.compile(r"[ \t\r\n0-9\-\.\,\:\;\(\)\[\]<>_/|~`'\"“”‘’·ㆍ]+")

def strip_parens(s: str) -> str:
    # 괄호류 안의 부가 정보를 제거
    return re.sub(r"\s*[$begin:math:text$\\[].*?[$end:math:text$\]]", "", s).strip()

def norm_comp_key(s: str) -> str:
    # 비교용 정규화: 공백/숫자/구두점 제거
    return NON_WORDS_RE.sub("", s)

def build_title_aliases(law_title: str):
    base = law_title.strip()
    no_paren = strip_parens(base)  # 예: "도로교통법 시행규칙"

    # 공백 유연 정규식(라인 일치용)
    def flex_spaces(text: str) -> re.Pattern:
        pat = re.escape(text.strip())
        pat = re.sub(r"\s+", r"\\s*", pat)
        return re.compile(r"^\s*" + pat + r"\s*$")

    aliases = {
        "raw": base,
        "no_paren": no_paren,
        "raw_norm": norm_comp_key(base),
        "no_paren_norm": norm_comp_key(no_paren),
        "raw_re": flex_spaces(base),
        "no_paren_re": flex_spaces(no_paren),
    }
    return aliases

def is_title_line(line: str, aliases: dict) -> bool:
    # (1) 정확/유사(공백 유연) 라인 일치
    if aliases["raw_re"].match(line) or aliases["no_paren_re"].match(line):
        return True
    # (2) 정규화 키로 완전 동일 (공백/숫자/구두점 제거 후 동일)
    nline = norm_comp_key(line)
    if nline and (nline == aliases["raw_norm"] or nline == aliases["no_paren_norm"]):
        return True
    # (3) 짧은 코어 제목이 한 줄로만 있는 경우
    core = aliases["no_paren_norm"]
    if len(core) >= 6 and nline == core:
        return True
    return False

# ---------- 제거/정규화 규칙 ----------
TOP_BRACKET_BLOCK_RE = re.compile(r"^\s*(?:$begin:math:display$[^$end:math:display$\n]+\]\s*){1,20}", flags=re.MULTILINE)


HEADER_FOOTER_RE = re.compile(
    "|".join([
        r"^\s*법제처.*국가법령정보센터.*$",
        r"^\s*국가법령정보센터.*$",
        r"^\s*www\.law\.go\.kr\s*$",
        r"^\s*페이지\s*\d+\s*$",
        r"^\s*[-–—―]+\s*$",
        r"^\s*\d+\s*$",
    ]),
    flags=re.IGNORECASE | re.MULTILINE,
)

PHONE_RE = re.compile(r"\b\d{2,3}-\d{3,4}-\d{4}\b")
ORG_HINT_RE = re.compile(r"(경찰청|행정안전부|법제처|담당부서|문의|전화|Fax|팩스|연락처)", re.IGNORECASE)
def is_contact_line(line: str) -> bool:
    return bool(PHONE_RE.search(line) and ORG_HINT_RE.search(line))

# --- 멀티라인까지 제거하는 메타 패턴 (실제 []/<> 기준) ---
META_KEYWORDS = r"(개정|전문개정|전부개정|일부개정|신설|본조신설|삭제|정정|준용|부칙|제목개정|이동|전단|후단|종전|현행)"
# 대괄호 전체 제거: [ ... META_KEYWORDS ... ]
BRACKET_META_ANY_RE = re.compile(r"\s*\[(?=[\s\S]*?" + META_KEYWORDS + r")[\s\S]*?\]\s*")
# 꺾쇠 전체 제거: < ... META_KEYWORDS ... >
ANGLE_META_ANY_RE   = re.compile(r"\s*<(?=[\s\S]*?" + META_KEYWORDS + r")[\s\S]*?>\s*")
# 깨진 대괄호 조각 방지(옵션)
BROKEN_BRACKET_META_RE = re.compile(r"$begin:math:display$\\s*(?=[\\s\\S]*?" + META_KEYWORDS + r")[^$end:math:display$]*")
# --- 장(章) 헤더 라인 제거 ---
# 예: "제2장 보행자의 통행방법" 같은 "한 줄" 전체 삭제
CHAPTER_HEADER_LINE_RE = re.compile(r"(?m)^\s*제\s*\d+\s*장(?:\s+[^\n]*)?\s*$")

# ---------- 핵심 패턴: "제 N조(" 형태의 '정식 조문 헤더'만 ----------
# - '제 1조의 …'는 (?!\s*의)로 배제
# - 반드시 '('가 바로 이어지는 헤더만 허용
ARTICLE_HEADER_INLINE = r"(제\s*\d+\s*조)(?!\s*의)\s*\("   # 인라인/라인 공통 탐지용
ARTICLE_HEADER_INLINE_RE = re.compile(ARTICLE_HEADER_INLINE)
ARTICLE_HEADER_LINE_RE   = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

# 본문 중간의 "제목 + 조 헤더" 패턴 제거 (제목만 제거)
def remove_inline_title_before_article(txt: str, law_title: str):
    core_title = strip_parens(law_title)
    title_inline_re = re.compile(r"(?m)(^|\n)\s*" + re.escape(core_title) + rf"\s+(?={ARTICLE_HEADER_INLINE})")
    return title_inline_re.sub("\n\n", txt)

# 본문 어디에 있어도 "제목 한 줄"이 단독으로 낀 경우 제거
def remove_lonely_title_lines(txt: str, aliases: dict):
    lines = txt.splitlines()
    kept = []
    for raw in lines:
        line = raw.strip()
        if not line:
            kept.append(raw)
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(raw)
    return "\n".join(kept)

# ---------- 본문 클리너 ----------
def clean_text_legal(text: str, law_title: str) -> str:
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # 상단 메타/헤더푸터 제거
    text = TOP_BRACKET_BLOCK_RE.sub("", text, count=1)
    text = HEADER_FOOTER_RE.sub("", text)

    aliases = build_title_aliases(law_title)

    # 줄 단위 기본 필터링(연락처/기관 안내/제목 라인)
    kept = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            kept.append("")
            continue
        if is_contact_line(line):
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(line)
    txt = "\n".join(kept)

    # 본문 중간 제목+조헤더 패턴 정리
    txt = remove_inline_title_before_article(txt, law_title)

    # --- (보호) '삭제<...>'는 남겨야 하므로 각도괄호를 임시 토큰으로 보호 ---
    def _protect_delete_angle(m):
        return m.group(0).replace("<", "«KEEP_ANGLE_OPEN»").replace(">", "«KEEP_ANGLE_CLOSE»")
    # '삭제<' 또는 '전부 삭제<' 변형까지 보호 (멀티라인)
    txt = re.sub(r"(?:전부\s*)?삭제\s*<[\s\S]*?>", _protect_delete_angle, txt)

    # --- 메타 제거: []·<> 안의 META_KEYWORDS 포함 블록은 전부 삭제 (멀티라인) ---
    txt = BRACKET_META_ANY_RE.sub(" ", txt)     # [본조신설 …], [제37조의2에서 이동 <…>] 등
    txt = ANGLE_META_ANY_RE.sub(" ", txt)       # <개정 …>, <… 이동 …> 등
    txt = BROKEN_BRACKET_META_RE.sub(" ", txt)  # 깨진 대괄호 조각도 정리

    # --- 보호 토큰 원복 ---
    txt = txt.replace("«KEEP_ANGLE_OPEN»", "<").replace("«KEEP_ANGLE_CLOSE»", ">")

    # 혹시 남은 단독 제목 라인 한 번 더 제거
    txt = remove_lonely_title_lines(txt, aliases)

    # ====== 조 헤더 앞/주변 개행 정규화 (엄격 헤더에만 적용) ======
    # 0) 본문 중간에 붙은 "… 제 N조(" 앞에 줄바꿈 1회 강제
    txt = re.sub(rf"(?!^)(?<!\n){ARTICLE_HEADER_INLINE}", r"\n\1(", txt)

    # 1) 조 헤더 앞 여러 빈 줄 → 정확히 1줄
    txt = re.sub(rf"(?m)\n{{2,}}(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)

    # 2) 문서 시작에서 곧바로 헤더가 오면 앞에 1줄 추가(가독)
    txt = re.sub(rf"(?m)\A(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)
    # --- 보호 토큰 원복 ---
    txt = txt.replace("«KEEP_ANGLE_OPEN»", "<").replace("«KEEP_ANGLE_CLOSE»", ">")

    # >>> 여기 한 줄 추가
    txt = CHAPTER_HEADER_LINE_RE.sub("", txt)

    # 혹시 남은 단독 제목 라인 한 번 더 제거
    txt = remove_lonely_title_lines(txt, aliases)

    # 하이픈 줄바꿈 제거 + 공백/개행 정리
    txt = re.sub(r"-\n", "", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    # 일반 빈 줄은 최대 1줄
    txt = re.sub(r"\n{3,}", "\n\n", txt)

    return txt.strip()

# ---------- 스플리팅: "제 N조(" 헤더 라인만 ----------
ARTICLE_SPLIT_RE = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

def split_into_articles_only_jo(text: str, law_title: str, source_meta: dict) -> List[Document]:
    matches = list(ARTICLE_SPLIT_RE.finditer(text))
    if not matches:
        return [Document(page_content=f"{law_title}\n{text}", metadata=source_meta)]

    articles: List[Document] = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if not chunk:
            continue
        # 조의 n(예: 제2조의2, 제14조의3 등)은 같은 블록 내부에 그대로 유지됨
        chunk = f"{law_title} {chunk}"
        # 블록 내부 여분 개행 정리
        chunk = re.sub(r"\n{3,}", "\n\n", chunk)
        articles.append(Document(page_content=chunk, metadata=source_meta))
    return articles

# ---------- 메인 ----------
if __name__ == "__main__":
    pdf_paths = sorted(glob(os.path.join(PDF_DIR, "*.pdf")))
    if MAX_FILES:
        pdf_paths = pdf_paths[:MAX_FILES]
    if not pdf_paths:
        raise FileNotFoundError(f"'{PDF_DIR}' 폴더에 PDF가 없습니다.")

    all_articles: List[Document] = []

    for p in tqdm(pdf_paths, desc="Processing PDFs"):
        loader = PyMuPDFLoader(p)
        pages = loader.load()

        law_title = normalize_title_from_source(p)
        page_texts = [clean_text_legal(d.page_content or "", law_title) for d in pages]
        # 페이지 합칠 때는 페이지 사이 1개 빈 줄만
        merged = "\n".join([t for t in page_texts if t.strip()])

        docs = split_into_articles_only_jo(merged, law_title, {"source": p, "law_title": law_title})
        all_articles.extend(docs)

    print(f"[INFO] 전처리/분할 완료. 조문 단위 문서 수: {len(all_articles)}")

    # 샘플 출력
    for i, d in enumerate(all_articles[:PRINT_SAMPLE_DOCS], start=1):
        print(f"\n--- 조문 샘플 {i} ---")
        print("law_title:", d.metadata.get("law_title"))
        print(d.page_content)#[:SNIPPET_CHARS])

In [ ]:
 # ===========================
# 법령 PDF 전처리 + "제 N조(…)" 헤더 기준 스플리팅
# - 참조 표현(예: "제 1조의 내용")에서는 절대 분할 X
# - 본문 중간에 붙은 "제 N조(…)"는 헤더로 인식해 줄바꿈 1회만 강제
# - []·<> 메타(개정/신설/이동/부칙 등) 전부 제거(멀티라인 포함)
# - 단, '삭제<YYYY. M. D.>' 형식은 반드시 보존
# ===========================
import os
import re
from glob import glob
from typing import List
from tqdm import tqdm

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

# ---------- 사용자 설정 ----------
PDF_DIR = "file/도로 교통법"   # 처리할 PDF 폴더
MAX_FILES = None               # None=전체, 숫자=일부
PRINT_SAMPLE_DOCS = 40         # 샘플 개수
SNIPPET_CHARS = 400            # 샘플 글자수

# ---------- 파일명 끝 (YYYYMMDD) 제거 ----------
TITLE_DATE_SUFFIX_RE = re.compile(r"\(\d{8}\)$")
def normalize_title_from_source(src: str) -> str:
    base = os.path.basename(src)
    name, _ = os.path.splitext(base)
    return TITLE_DATE_SUFFIX_RE.sub("", name).strip()

# ---------- 제목/별칭 생성 & 비교 유틸 ----------
NON_WORDS_RE = re.compile(r"[ \t\r\n0-9\-\.\,\:\;\(\)\[\]<>_/|~`'\"“”‘’·ㆍ]+")

def strip_parens(s: str) -> str:
    # 괄호류 안의 부가 정보를 제거
    return re.sub(r"\s*[$begin:math:text$\\[].*?[$end:math:text$\]]", "", s).strip()

def norm_comp_key(s: str) -> str:
    # 비교용 정규화: 공백/숫자/구두점 제거
    return NON_WORDS_RE.sub("", s)

def build_title_aliases(law_title: str):
    base = law_title.strip()
    no_paren = strip_parens(base)  # 예: "도로교통법 시행규칙"

    # 공백 유연 정규식(라인 일치용)
    def flex_spaces(text: str) -> re.Pattern:
        pat = re.escape(text.strip())
        pat = re.sub(r"\s+", r"\\s*", pat)
        return re.compile(r"^\s*" + pat + r"\s*$")

    aliases = {
        "raw": base,
        "no_paren": no_paren,
        "raw_norm": norm_comp_key(base),
        "no_paren_norm": norm_comp_key(no_paren),
        "raw_re": flex_spaces(base),
        "no_paren_re": flex_spaces(no_paren),
    }
    return aliases

def is_title_line(line: str, aliases: dict) -> bool:
    # (1) 정확/유사(공백 유연) 라인 일치
    if aliases["raw_re"].match(line) or aliases["no_paren_re"].match(line):
        return True
    # (2) 정규화 키로 완전 동일 (공백/숫자/구두점 제거 후 동일)
    nline = norm_comp_key(line)
    if nline and (nline == aliases["raw_norm"] or nline == aliases["no_paren_norm"]):
        return True
    # (3) 짧은 코어 제목이 한 줄로만 있는 경우
    core = aliases["no_paren_norm"]
    if len(core) >= 6 and nline == core:
        return True
    return False

# ---------- 제거/정규화 규칙 ----------
TOP_BRACKET_BLOCK_RE = re.compile(r"^\s*(?:$begin:math:display$[^$end:math:display$\n]+\]\s*){1,20}", flags=re.MULTILINE)


HEADER_FOOTER_RE = re.compile(
    "|".join([
        r"^\s*법제처.*국가법령정보센터.*$",
        r"^\s*국가법령정보센터.*$",
        r"^\s*www\.law\.go\.kr\s*$",
        r"^\s*페이지\s*\d+\s*$",
        r"^\s*[-–—―]+\s*$",
        r"^\s*\d+\s*$",
        r"도로교통법 시행규칙"
    ]),
    flags=re.IGNORECASE | re.MULTILINE,
)

PHONE_RE = re.compile(r"\b\d{2,3}-\d{3,4}-\d{4}\b")
ORG_HINT_RE = re.compile(r"(경찰청|행정안전부|법제처|담당부서|문의|전화|Fax|팩스|연락처)", re.IGNORECASE)
def is_contact_line(line: str) -> bool:
    return bool(PHONE_RE.search(line) and ORG_HINT_RE.search(line))

# --- 멀티라인까지 제거하는 메타 패턴 (실제 []/<> 기준) ---
META_KEYWORDS = r"(개정|전문개정|전부개정|일부개정|신설|본조신설|삭제|정정|준용|부칙|제목개정|이동|전단|후단|종전|현행)"
# 대괄호 전체 제거: [ ... META_KEYWORDS ... ]
BRACKET_META_ANY_RE = re.compile(r"\s*\[(?=[\s\S]*?" + META_KEYWORDS + r")[\s\S]*?\]\s*")
# 꺾쇠 전체 제거: < ... META_KEYWORDS ... >
ANGLE_META_ANY_RE   = re.compile(r"\s*<(?=[\s\S]*?" + META_KEYWORDS + r")[\s\S]*?>\s*")
# 깨진 대괄호 조각 방지(옵션)
BROKEN_BRACKET_META_RE = re.compile(r"$begin:math:display$\\s*(?=[\\s\\S]*?" + META_KEYWORDS + r")[^$end:math:display$]*")
# --- 장(章) 헤더 라인 제거 ---
# 예: "제2장 보행자의 통행방법" 같은 "한 줄" 전체 삭제
CHAPTER_HEADER_LINE_RE = re.compile(r"(?m)^\s*제\s*\d+\s*장(?:\s+[^\n]*)?\s*$")

# ---------- 핵심 패턴: "제 N조(" 형태의 '정식 조문 헤더'만 ----------
# - '제 1조의 …'는 (?!\s*의)로 배제
# - 반드시 '('가 바로 이어지는 헤더만 허용
ARTICLE_HEADER_INLINE = r"(제\s*\d+\s*조)(?!\s*의)\s*\("   # 인라인/라인 공통 탐지용
ARTICLE_HEADER_INLINE_RE = re.compile(ARTICLE_HEADER_INLINE)
ARTICLE_HEADER_LINE_RE   = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

# 본문 중간의 "제목 + 조 헤더" 패턴 제거 (제목만 제거)
def remove_inline_title_before_article(txt: str, law_title: str):
    core_title = strip_parens(law_title)
    title_inline_re = re.compile(r"(?m)(^|\n)\s*" + re.escape(core_title) + rf"\s+(?={ARTICLE_HEADER_INLINE})")
    return title_inline_re.sub("\n\n", txt)

# 본문 어디에 있어도 "제목 한 줄"이 단독으로 낀 경우 제거
def remove_lonely_title_lines(txt: str, aliases: dict):
    lines = txt.splitlines()
    kept = []
    for raw in lines:
        line = raw.strip()
        if not line:
            kept.append(raw)
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(raw)
    return "\n".join(kept)

# ---------- 본문 클리너 ----------
def clean_text_legal(text: str, law_title: str) -> str:
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # 상단 메타/헤더푸터 제거
    text = TOP_BRACKET_BLOCK_RE.sub("", text, count=1)
    text = HEADER_FOOTER_RE.sub("", text)

    aliases = build_title_aliases(law_title)

    # 줄 단위 기본 필터링(연락처/기관 안내/제목 라인)
    kept = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            kept.append("")
            continue
        if is_contact_line(line):
            continue
        if is_title_line(line, aliases):
            continue
        kept.append(line)
    txt = "\n".join(kept)

    # 본문 중간 제목+조헤더 패턴 정리
    txt = remove_inline_title_before_article(txt, law_title)

    # --- (보호) '삭제<...>'는 남겨야 하므로 각도괄호를 임시 토큰으로 보호 ---
    def _protect_delete_angle(m):
        return m.group(0).replace("<", "«KEEP_ANGLE_OPEN»").replace(">", "«KEEP_ANGLE_CLOSE»")
    # '삭제<' 또는 '전부 삭제<' 변형까지 보호 (멀티라인)
    txt = re.sub(r"(?:전부\s*)?삭제\s*<[\s\S]*?>", _protect_delete_angle, txt)

    # --- 메타 제거: []·<> 안의 META_KEYWORDS 포함 블록은 전부 삭제 (멀티라인) ---
    txt = BRACKET_META_ANY_RE.sub(" ", txt)     # [본조신설 …], [제37조의2에서 이동 <…>] 등
    txt = ANGLE_META_ANY_RE.sub(" ", txt)       # <개정 …>, <… 이동 …> 등
    txt = BROKEN_BRACKET_META_RE.sub(" ", txt)  # 깨진 대괄호 조각도 정리

    # --- 보호 토큰 원복 ---
    txt = txt.replace("«KEEP_ANGLE_OPEN»", "<").replace("«KEEP_ANGLE_CLOSE»", ">")

    # 혹시 남은 단독 제목 라인 한 번 더 제거
    txt = remove_lonely_title_lines(txt, aliases)

    # ====== 조 헤더 앞/주변 개행 정규화 (엄격 헤더에만 적용) ======
    # 0) 본문 중간에 붙은 "… 제 N조(" 앞에 줄바꿈 1회 강제
    txt = re.sub(rf"(?!^)(?<!\n){ARTICLE_HEADER_INLINE}", r"\n\1(", txt)

    # 1) 조 헤더 앞 여러 빈 줄 → 정확히 1줄
    txt = re.sub(rf"(?m)\n{{2,}}(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)

    # 2) 문서 시작에서 곧바로 헤더가 오면 앞에 1줄 추가(가독)
    txt = re.sub(rf"(?m)\A(\s*{ARTICLE_HEADER_INLINE})", r"\n\1", txt)
    # --- 보호 토큰 원복 ---
    txt = txt.replace("«KEEP_ANGLE_OPEN»", "<").replace("«KEEP_ANGLE_CLOSE»", ">")

    # >>> 여기 한 줄 추가
    txt = CHAPTER_HEADER_LINE_RE.sub("", txt)

    # 혹시 남은 단독 제목 라인 한 번 더 제거
    txt = remove_lonely_title_lines(txt, aliases)

    # 하이픈 줄바꿈 제거 + 공백/개행 정리
    txt = re.sub(r"-\n", "", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    # 일반 빈 줄은 최대 1줄
    txt = re.sub(r"\n{3,}", "\n\n", txt)

    return txt.strip()

# ---------- 스플리팅: "제 N조(" 헤더 라인만 ----------
ARTICLE_SPLIT_RE = re.compile(rf"(?m)^\s*{ARTICLE_HEADER_INLINE}[^\n]*")

def split_into_articles_only_jo(text: str, law_title: str, source_meta: dict) -> List[Document]:
    matches = list(ARTICLE_SPLIT_RE.finditer(text))
    if not matches:
        return [Document(page_content=f"{law_title}\n{text}", metadata=source_meta)]

    articles: List[Document] = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if not chunk:
            continue
        # 조의 n(예: 제2조의2, 제14조의3 등)은 같은 블록 내부에 그대로 유지됨
        chunk = f"{law_title} {chunk}"
        # 블록 내부 여분 개행 정리
        chunk = re.sub(r"\n{3,}", "\n\n", chunk)
        articles.append(Document(page_content=chunk, metadata=source_meta))
    return articles

# ---------- 메인 ----------
if __name__ == "__main__":
    pdf_paths = sorted(glob(os.path.join(PDF_DIR, "*.pdf")))
    if MAX_FILES:
        pdf_paths = pdf_paths[:MAX_FILES]
    if not pdf_paths:
        raise FileNotFoundError(f"'{PDF_DIR}' 폴더에 PDF가 없습니다.")

    all_articles: List[Document] = []

    for p in tqdm(pdf_paths, desc="Processing PDFs"):
        loader = PyMuPDFLoader(p)
        pages = loader.load()

        law_title = normalize_title_from_source(p)
        page_texts = [clean_text_legal(d.page_content or "", law_title) for d in pages]
        # 페이지 합칠 때는 페이지 사이 1개 빈 줄만
        merged = "\n".join([t for t in page_texts if t.strip()])

        docs = split_into_articles_only_jo(merged, law_title, {"source": p, "law_title": law_title})
        all_articles.extend(docs)

    print(f"[INFO] 전처리/분할 완료. 조문 단위 문서 수: {len(all_articles)}")

    # 샘플 출력
    for i, d in enumerate(all_articles[:PRINT_SAMPLE_DOCS], start=1):
        print(f"\n--- 조문 샘플 {i} ---")
        print("law_title:", d.metadata.get("law_title"))
        print(d.page_content)#[:SNIPPET_CHARS])

In [ ]:
from retriever_factory import get_dense_retriever

retriever = get_dense_retriever(device="mps", top_k=10)
docs = retriever.invoke("어린이 보호구역의 제한 속도를 알려줘")

In [ ]:
docs

In [ ]:
!pip install vllm

In [ ]:
python -m vllm.entrypoints.openai.api_server --model deepseek-ai/DeepSeek-OCR

In [ ]:
import vllm

In [ ]:
from vllm import LLM, SamplingParams
import torch
import sys, os

# MPS 체크
use_mps = torch.backends.mps.is_available()
device_type = "mps" if use_mps else "cpu"

print(f"Using device: {device_type}")

# 작은 모델 사용 권장 (예: TinyLlama)
llm = LLM(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    download_dir="./models",
    tensor_parallel_size=1,
    trust_remote_code=True,
    dtype="float16" if use_mps else "float32"
)

sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=100)

prompt = "Write a short poem about artificial intelligence."
outputs = llm.generate([prompt], sampling_params)

for output in outputs:
    print(output.outputs[0].text)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "kfkas/Legal-Llama-2-ko-7b-Chat"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# MPS가 있으면 float16 + mps, 없으면 float32 + cpu
if torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
)
model.to(device)

prompt = "대한민국 헌법 제10조 내용을 요약해줘."

inputs = tokenizer(prompt, return_tensors="pt").to(device)
output_ids = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

In [ ]:
import torch
from langchain_community.vectorstores import Chroma
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from retriever_factory import get_dense_retriever


RAG_PROMPT_TEMPLATE = (
"""
절대 한국어로만 답변하십시오. 영어를 사용하지 마십시오.  
You must always respond in Korean. Do not use English.  

당신은 대한민국 현행 법령을 기반으로 답변하는 RAG 챗봇입니다.  
아래 지침을 따르고, 명확하고 간결한 답변을 제공하세요.

- 질문이 단순하면 한 문장으로 답변하세요.  
- 질문이 복잡하면 2~3문장 이내로 답변하세요.  
- 법 조항을 직접 나열하지 말고, 질문과 관련된 핵심 내용을 요약하여 자연스럽게 설명하세요.  
- 각 문장에 해당하는 법적 근거 조항을 괄호 안에 정확히 명시하세요. (예: 도로교통법 제12조 제1항)  
- 질문을 반복하지 말고, 불필요한 형식적인 표현을 포함하지 마세요.  
- **정확도, 확률, 신뢰도 같은 수치를 답변에 절대 포함하지 마세요.**  
- Context 내에서만 답변을 생성하고, 모르는 내용은 `"제가 제공할 수 있는 정보가 없습니다."`라고만 답변하세요.  

{question}에 대한 답변을 다음 Context를 참고하여 작성하세요.

Context:  
{context}  

답변:
"""
)

SYS_PROMPT_TEMPLATE = (
"""
당신은 대한민국 현행 법령을 기반으로 작동하는 RAG 챗봇입니다.  
절대 한국어로만 답변하십시오. 영어를 사용하지 마십시오.  

답변을 생성할 때, 다음 원칙을 따르세요.

1. **간결한 답변 제공**  
   - 단순한 질문 → 한 문장  
   - 복잡한 질문 → 2~3문장  
   - 불필요한 설명, 배경 정보, 반복 표현 금지  

2. **법 조항 요약 방식**  
   - 법 조항을 직접 나열하지 않고 질문과 관련된 핵심 내용을 요약  
   - 각 문장에 해당하는 법적 근거 조항을 괄호 안에 명확히 표기 (예: 도로교통법 제17조 제2항)

3. **불필요한 정보 제한**  
   - "정확도", "신뢰도", "확률" 등의 수치는 절대 포함 금지  
   - "또한", "추가로", "이 외에도"와 같은 확장 표현 사용 금지  

4. **추론 및 임의 해석 금지**  
   - 제공된 Context 범위 내에서만 답변 작성  
   - 필요한 정보가 없으면 `"제가 제공할 수 있는 정보가 없습니다."`라고만 답변  
"""
)

DEVICE = "cpu"

def setup_rag_system(llm_base_url):
    llm = ChatOpenAI(
        base_url=llm_base_url,
        api_key="lm-studio",
        model="teddylee777/EEVE-Korean-Instruct-10.8B-v1.0-gguf",
        temperature=0.1,
    )
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", SYS_PROMPT_TEMPLATE),
        ("user", RAG_PROMPT_TEMPLATE),
    ])
    rag_chain = prompt_template | llm | StrOutputParser()
    return llm, rag_chain


def run_pipeline(question, llm_base_url="http://localhost:1234/v1", chroma_dir="chroma_store"):
    if not question.strip():
        return "질문을 입력해주세요.", [], []

    retriever = get_dense_retriever(device="mps", top_k=10)
    retrieved_docs = retriever.invoke(question)
    if not retrieved_docs:
        return "제가 제공할 수 있는 정보가 없습니다.", [], []

    context = " ".join(doc.page_content for doc in retrieved_docs)
    llm, rag_chain = setup_rag_system(llm_base_url)
    response = rag_chain.invoke({"question": question, "context": context})
    return response, retrieved_docs


In [ ]:
run_pipeline("어린이 보호 구역 제한 속도를 알려주세요")

In [ ]:
import os
from typing import List
from openai import OpenAI

from retriever_factory import get_dense_retriever  # 네가 만든 함수

# =========================
# 프롬프트 템플릿
# =========================
RAG_PROMPT_TEMPLATE = (
"""
절대 한국어로만 답변하십시오. 영어를 사용하지 마십시오.  
You must always respond in Korean. Do not use English.  

당신은 대한민국 현행 법령을 기반으로 답변하는 RAG 챗봇입니다.  
아래 지침을 따르고, 명확하고 간결한 답변을 제공하세요.

- 질문이 단순하면 한 문장으로 답변하세요.  
- 질문이 복잡하면 2~3문장 이내로 답변하세요.  
- 법 조항을 직접 나열하지 말고, 질문과 관련된 핵심 내용을 요약하여 자연스럽게 설명하세요.  
- 각 문장에 해당하는 법적 근거 조항을 괄호 안에 정확히 명시하세요. (예: 도로교통법 제12조 제1항)  
- 질문을 반복하지 말고, 불필요한 형식적인 표현을 포함하지 마세요.  
- **정확도, 확률, 신뢰도 같은 수치를 답변에 절대 포함하지 마세요.**  
- Context 내에서만 답변을 생성하고, 모르는 내용은 `"제가 제공할 수 있는 정보가 없습니다."`라고만 답변하세요.  

{question}에 대한 답변을 다음 Context를 참고하여 작성하세요.

Context:  
{context}  

답변:
"""
)

SYS_PROMPT_TEMPLATE = (
"""
당신은 대한민국 현행 법령을 기반으로 작동하는 RAG 챗봇입니다.  
절대 한국어로만 답변하십시오. 영어를 사용하지 마십시오.  

답변을 생성할 때, 다음 원칙을 따르세요.

1. **간결한 답변 제공**  
   - 단순한 질문 → 한 문장  
   - 복잡한 질문 → 2~3문장  
   - 불필요한 설명, 배경 정보, 반복 표현 금지  

2. **법 조항 요약 방식**  
   - 법 조항을 직접 나열하지 않고 질문과 관련된 핵심 내용을 요약  
   - 각 문장에 해당하는 법적 근거 조항을 괄호 안에 명확히 표기 (예: 도로교통법 제17조 제2항)

3. **불필요한 정보 제한**  
   - "정확도", "신뢰도", "확률" 등의 수치는 절대 포함 금지  
   - "또한", "추가로", "이 외에도"와 같은 확장 표현 사용 금지  

4. **추론 및 임의 해석 금지**  
   - 제공된 Context 범위 내에서만 답변 작성  
   - 필요한 정보가 없으면 `"제가 제공할 수 있는 정보가 없습니다."`라고만 답변  
"""
)

# =========================
# LM Studio용 클라이언트
# =========================
def get_lmstudio_client(base_url: str = "http://localhost:1234/v1") -> OpenAI:
    """
    LM Studio는 OpenAI 호환 API를 쓰니까,
    base_url + api_key="lm-studio"로 호출해주면 됨.
    """
    client = OpenAI(
        base_url=base_url,
        api_key="lm-studio",
    )
    return client

# =========================
# LLM 호출 함수
# =========================
def call_llm_with_context(
    question: str,
    context: str,
    base_url: str = "http://localhost:1234/v1",
    model_name: str = "teddylee777/EEVE-Korean-Instruct-10.8B-v1.0-gguf",
) -> str:
    client = get_lmstudio_client(base_url)

    user_content = RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context,
    )

    messages = [
        {"role": "system", "content": SYS_PROMPT_TEMPLATE},
        {"role": "user", "content": user_content},
    ]

    completion = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.1,
    )
    return completion.choices[0].message.content

# =========================
# 전체 파이프라인
# =========================
def run_pipeline(
    question: str,
    llm_base_url: str = "http://localhost:1234/v1",
    top_k: int = 10,
    device: str = "cpu",
):
    """
    - retriever_factory.get_dense_retriever() 사용
    - re-ranker 없이 retriever 결과를 그대로 context로 사용
    - 반환: (답변 텍스트, retrieved_docs, used_docs)
    """
    if not question.strip():
        return "질문을 입력해주세요.", [], []

    # 1) Dense retriever 로드 (네가 만든 팩토리 함수 사용)
    retriever = get_dense_retriever(device=device, top_k=top_k)

    # 2) 검색
    retrieved_docs = retriever.invoke(question)
    if not retrieved_docs:
        return "제가 제공할 수 있는 정보가 없습니다.", [], []

    # 3) Context 생성 (필요하면 메타데이터도 같이 붙여도 됨)
    #    일단은 내용만 합치는 단순 버전
    context_chunks: List[str] = []
    for d in retrieved_docs:
        context_chunks.append(d.page_content)
    context = "\n\n".join(context_chunks)

    # 4) LLM 호출
    response = call_llm_with_context(
        question=question,
        context=context,
        base_url=llm_base_url,
    )

    # 인터페이스 맞추려고 세 번째 값도 retrieved_docs 그대로
    return response, retrieved_docs, retrieved_docs

# =========================
# 간단 실행 예시
# =========================
if __name__ == "__main__":
    q = "어린이 보호구역의 제한 속도를 알려줘"
    answer, docs, used = run_pipeline(
        question=q,
        llm_base_url="http://localhost:1234/v1",
        top_k=10,
        device="cpu",   # M1이면 "mps"로 retriever 올려도 됨
    )

    print("=== 답변 ===")
    print(answer)
    print("\n=== 사용된 문서 수 ===", len(docs))

In [ ]:
answer, docs, used = run_pipeline("음주운전의 처벌 기준을 알려주세요")
print(answer)

In [ ]:
print(docs)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 1) 모델 이름
model_name = "kfkas/Legal-Llama-2-ko-7b-Chat"

# 2) 토크나이저 / 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Mac MPS 사용 여부 확인
if torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
)
model.to("cpu")

# 3) 프롬프트 준비 (LLaMA2-Chat 스타일)
system_prompt = "너는 대한민국 현행 법령을 정확하게 설명하는 한국어 법률 AI 비서이다."
user_prompt = "도로교통법 제 11조가 무슨 내용인지 알려줘."

prompt = f"<s>[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n{user_prompt} [/INST]"

# 4) 토크나이즈 → 모델 입력
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# 5) 추론(generate)
output_ids = model.generate(
    **inputs,
    max_new_tokens=16,
    temperature=0.3,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

# 6) 텍스트 디코딩
response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("\n=== 모델 응답 ===\n")
print(response)

In [ ]:
import torch
print("MPS available:", torch.backends.mps.is_available())
print("Device using:", device)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, time

model_name = "kfkas/Legal-Llama-2-ko-7b-Chat"

tokenizer = AutoTokenizer.from_pretrained(model_name)

device = "cpu"
dtype = torch.float32   # CPU는 fp32가 안정적

start = time.time()
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
)
model.to(device)
print("모델 로드 완료:", time.time() - start, "초")

system_prompt = "너는 대한민국 현행 법령을 설명하는 한국어 법률 AI 비서이다."
user_prompt = "도로교통법 제 11조가 무슨 내용인지 요약해줘."

prompt = f"<s>[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n{user_prompt} [/INST]"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

start = time.time()
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=16,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
print("generate 시간:", time.time() - start, "초")

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

In [ ]:
from pipeline import run_pipeline

answer, docs, used = run_pipeline("어린의 보호 구역의 제한 속도는?")
print(answer)

In [ ]:
print(answer)

In [ ]:
docs

In [ ]:
used

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"
EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
DEVICE = "mps"   # 맥북이면 보통 "mps", 안되면 "cpu"
TOP_K = 5

# =========================
# 임베딩 클래스
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# =========================
# 전체 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])

documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

print(f"전체 문서 수: {len(docs)}")

# =========================
# BM25 Retriever 생성
# =========================
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = TOP_K

print("BM25 Retriever 생성 완료")

# =========================
# 테스트 질의
# =========================
query = "어린이 보호구역 제한속도는 얼마인가?"

results = bm25_retriever.invoke(query)

print(f"\n질문: {query}")
print(f"검색 결과 개수: {len(results)}\n")

for i, doc in enumerate(results, 1):
    print("=" * 100)
    print(f"[{i}] metadata")
    print(doc.metadata)
    print("-" * 100)
    print(doc.page_content[:1000])   # 너무 길면 앞부분만 보기
    print()

In [ ]:
import pandas as pd

evaluation_rows = [
    {
        "질문": "어린이 보호구역 제한속도는 얼마인가?",
        "Dense 평가": "핵심 조문인 도로교통법 제12조를 상위에 잘 검색함.",
        "BM25 평가": "보호구역 관련 문서를 비교적 잘 찾았고 핵심 조문도 포함됨.",
        "RRF 평가": "Dense와 BM25 공통 문서를 강화해 제12조를 상위로 올림.",
        "Reranker 평가": "제12조를 최종 1위로 유지하여 안정적으로 정렬함.",
        "최종 결론": "최종 파이프라인이 가장 잘 동작한 질문 중 하나.",
        "남은 문제": "주변 보호구역 관련 문서가 일부 섞일 수 있음."
    },
    {
        "질문": "스쿨존 속도위반 기준은?",
        "Dense 평가": "어린이 보호구역 관련 조문 일부를 찾았지만 속도 기준과 직접 연결은 약했음.",
        "BM25 평가": "기준 같은 일반 토큰에 끌려 관련 없는 기준 조문이 많이 검색됨.",
        "RRF 평가": "Dense 문서를 어느 정도 유지했지만 BM25 노이즈도 함께 반영됨.",
        "Reranker 평가": "핵심 조문을 충분히 위로 올리지 못했고 제17조도 최상위로 끌어올리지 못함.",
        "최종 결론": "최종 파이프라인에서도 한계가 남은 실패 사례.",
        "남은 문제": "스쿨존과 어린이 보호구역 표현 차이 해결 필요."
    },
    {
        "질문": "주정차 위반 과태료 기준은?",
        "Dense 평가": "과태료 관련 문서를 비교적 잘 찾았음.",
        "BM25 평가": "과태료/부과 관련 문서를 잘 수집했으나 행정 절차 문서도 섞임.",
        "RRF 평가": "관련 후보군을 잘 모았지만 최상위가 꼭 기준 조문은 아니었음.",
        "Reranker 평가": "시행령의 과태료 부과기준 문서를 최상위로 올려 의미 있게 개선됨.",
        "최종 결론": "reranker 효과가 분명히 나타난 부분 성공 사례.",
        "남은 문제": "top-k 전체는 여전히 부과 절차/행정 문서 노이즈가 남아 있음."
    },
    {
        "질문": "음주운전 면허취소 기준은?",
        "Dense 평가": "관련 조문 후보는 찾았지만 최상위가 다소 빗나간 경우가 있었음.",
        "BM25 평가": "운전면허 취소/정지 관련 문서를 일부 수집했지만 일반 면허 문서로도 퍼졌음.",
        "RRF 평가": "공통 문서 강화는 되었으나 핵심 조문 최상위화는 부족했음.",
        "Reranker 평가": "제93조와 제91조 같은 핵심 조문을 최상위권으로 올려 가장 큰 개선 효과를 보임.",
        "최종 결론": "reranker 필요성을 가장 잘 보여주는 대표 성공 사례.",
        "남은 문제": "일부 이상 청크가 후보군에 끼는 문제는 남아 있음."
    },
    {
        "질문": "불법주차 신고 가능 여부는?",
        "Dense 평가": "주차위반 조치, 견인 관련 문서를 부분적으로 잘 찾았음.",
        "BM25 평가": "불법, 신고 같은 일반 토큰에 끌려 불법부착장치 등 엉뚱한 문서가 섞였음.",
        "RRF 평가": "제35조, 견인 조치 문서를 상위에 올리며 어느 정도 보완함.",
        "Reranker 평가": "일부 좋은 문서는 유지했지만 불법부착장치 같은 노이즈 문서도 상위에 남겼음.",
        "최종 결론": "부분 개선은 있었지만 최종 품질은 아직 불안정한 질문.",
        "남은 문제": "query semantics 문제와 후보군 노이즈가 큼."
    }
]

df_eval = pd.DataFrame(evaluation_rows)
df_eval.to_csv("file/csv/manual_evaluation_table.csv", index=False, encoding="utf-8-sig")